# Transformer：从论文公式到 PyTorch 实现

> **本章定位**：本章以《Attention Is All You Need》为理论基线，集中讨论原始 Encoder–Decoder Transformer，并在 2.9 节建立 Causal / Non-causal 与三类主流 Transformer 架构的坐标；Decoder-only、Causal LM 和现代 GPT 的实现细节见 `31_nlp_gpt.ipynb`。

本章从无上下文的逐位置预测基线出发，依次加入 Token Embedding、正弦位置编码、Scaled Dot-Product Attention、Mask、Multi-Head Attention、FFN、残差连接和 LayerNorm，再组装 Encoder、Decoder 与 Cross-Attention。最后使用 Tatoeba 中译英句对连通 Teacher Forcing、训练和自回归解码，与 `torch.nn.Transformer` 做形状、参数和数值契约对齐，并映射到主流 MarianMT 生产接口。

下图先按《Attention Is All You Need》给出原始架构：Encoder 与 Decoder 各堆叠 6 层，每个子层之后执行残差相加与 LayerNorm，即 Post-LN。后文先逐项实现这些组件，再在 3.3 节说明本章训练为何切换到 Pre-LN，并在第 5 节映射到 `torch.nn.Transformer`。

<!-- diagram:transformer-paper-architecture -->

![架构图：原始 Transformer Encoder–Decoder 论文架构与张量形状](assets/figures/30_transformer/transformer-paper-architecture.svg)

[TikZ 源文件](assets/figures/30_transformer/transformer-paper-architecture.tex)

本章实现原始 Transformer 的 Encoder–Decoder 机制，并先说明它与 Encoder-only、Decoder-only 的关系；GPT 对 Encoder 和 Cross-Attention 的移除，以及 Decoder-only 主干的完整实现，由下一章展开。


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 共同基础：Transformer 机制 |
| 本章定位 | 以《Attention Is All You Need》为主线，集中学习 Encoder–Decoder Transformer 的核心理论。 |
| 先修知识 | 完成 `10`、`21`；理解矩阵乘法、Softmax 和基本 PyTorch 张量操作。 |
| 预计时间 | 3～4 小时 |
| 运行资源 | CPU、Apple Silicon 或 Colab；原理训练从 Hugging Face Hub 下载约 1.49 MiB 语料，生产迁移首次下载约 312 MB MarianMT 权重。 |
| 输入 | 中英句对、token ID、padding mask 与 causal mask。 |
| 交付物 | Encoder–Decoder 原理实现、翻译闭环和 `torch.nn.Transformer` 数值契约。 |


### 1.1．学习目标

完成本章后，读者能够区分 Causal / Non-causal 注意力可见性与 Encoder-only / Decoder-only / Encoder–Decoder 架构组成，推导 Encoder–Decoder Transformer 的核心公式，从零实现 Attention、FFN、残差与归一化模块，构建 Teacher Forcing 翻译训练闭环，并与 `torch.nn.Transformer` 完成形状、参数和数值对照。

### 1.2．环境与依赖

本章依赖 PyTorch、Transformers、SentencePiece、SacreMoses、Datasets、Hugging Face Hub、Matplotlib 与 Safetensors，具体版本由项目依赖文件统一管理。

In [ ]:
# 环境：Python 3.11+，PyTorch 2.3+，Transformers 5.13+，Matplotlib 3.8+
# 若环境缺少依赖，请先取消下一行注释并执行：
# %pip install -U "transformers>=5.13,<6" sentencepiece sacremoses datasets huggingface_hub torch matplotlib safetensors

import math
import random
from dataclasses import dataclass
from functools import lru_cache

import transformers
from datasets import load_dataset
from huggingface_hub import hf_hub_download

import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

# 固定批次抽样与参数初始化；质量比较需使用预先登记的多个种子。
SEED = 42  # 仅支持实验重放，不保证跨设备或库版本逐位一致。
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"device: {DEVICE}")

## 2．直觉与输入输出契约

### 2.1．用于训练闭环的中英句对

本 notebook 独立从 Hugging Face Hub 下载 Tatoeba 中英平行语料，不读取其他 notebook 生成的数据、Tokenizer、Checkpoint 或 Manifest。数据在这里仅为 Transformer 提供真实的变长序列：加载两列非空文本，Tokenize 后过滤超过位置编码上限的句对，再按固定随机种子划分训练集、验证集和测试集。

| 步骤 | 本章保留的原因 |
|---|---|
| 加载中英句对 | 为 Encoder–Decoder 提供真实源序列和目标序列。 |
| 过滤空文本与超长序列 | 避免空批次，并保证位置编码覆盖全部 token。 |
| 固定随机划分 | 区分参数更新、checkpoint 选择和最终行为观察。 |

Tatoeba 短句只用于观察 Teacher Forcing、自回归生成和 Cross-Attention，不代表通用翻译质量。

In [ ]:
# 独立下载真实中英句对；本 notebook 不依赖其他章节生成的数据文件。

DATASET_ID = "Helsinki-NLP/tatoeba_mt"
DATASET_FILENAME = "dev/tatoeba-dev.eng-cmn_Hans.tsv"
MODEL_MAX_LENGTH = 128  # 实验序列上限；普通 Attention 的成本随长度近似平方增长。

dataset_path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=DATASET_FILENAME,
)
raw_dataset = load_dataset(
    "csv",
    data_files={"source": dataset_path},
    delimiter="\t",
    column_names=["source_lang", "target_lang", "english", "chinese"],
    split="source",
)

RAW_PAIRS = [
    (chinese.strip(), english.strip())
    for chinese, english in zip(raw_dataset["chinese"], raw_dataset["english"])
    if isinstance(chinese, str)
    and isinstance(english, str)
    and chinese.strip()
    and english.strip()
]

print(f"Tatoeba 中英句对：{len(RAW_PAIRS):,}")
print("样例：", RAW_PAIRS[0])

### 2.2．复用 Hugging Face 官方 ByT5 Tokenizer

本 notebook 独立下载并加载 `google/byt5-small` 的字节级 Tokenizer，让中文、英文和未登录字符共享一个较小词表。为了保持 Transformer 教学主线，这里只使用文本与 token ID 的转换接口，不展开 Tokenizer 训练过程。

ByT5 没有独立 BOS。本章只复用 Tokenizer、不加载 ByT5 模型权重，因此显式新增 `<s>` 作为 Decoder 起始 token。新增后词表扩大一项，Embedding 与 LM Head 都使用更新后的 `VOCAB_SIZE`；PAD、BOS 和 EOS 使用不同 ID，避免目标 Padding Mask 屏蔽 Decoder 的第一个位置。

In [ ]:
# 加载 Tokenizer，并提供本章后续模型代码需要的最小编码接口。

from transformers import AutoTokenizer

TOKENIZER_ID = "google/byt5-small"
principle_tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    use_fast=False,
    trust_remote_code=False,
)
principle_tokenizer.add_special_tokens({"bos_token": "<s>"})
principle_tokenizer.model_max_length = MODEL_MAX_LENGTH

PAD_ID = int(principle_tokenizer.pad_token_id)
BOS_ID = int(principle_tokenizer.bos_token_id)
EOS_ID = int(principle_tokenizer.eos_token_id)
VOCAB_SIZE = len(principle_tokenizer)
SPECIAL_TOKEN_TO_ID = {"<pad>": PAD_ID, "<s>": BOS_ID, "</s>": EOS_ID}


@lru_cache(maxsize=None)
def my_encode_content(text):
    """缓存不含控制 token 的编码，避免训练各 epoch 重复执行 Tokenize。"""
    return tuple(principle_tokenizer.encode(text, add_special_tokens=False, verbose=False))


def my_encode(text, add_bos=False, add_eos=False):
    """把一条文本编码为 token ID，并按需添加序列控制 token。"""
    token_ids = list(my_encode_content(text))
    return ([BOS_ID] if add_bos else []) + token_ids + ([EOS_ID] if add_eos else [])


def my_batch_encode(texts, add_bos=False, add_eos=False, device=None):
    """批量编码并右侧补齐文本，返回形状为 [N, S] 的 token 张量。"""
    rows = [
        {"input_ids": my_encode(text, add_bos=add_bos, add_eos=add_eos)}
        for text in texts
    ]
    if not rows:
        raise ValueError("至少需要一条文本")
    if max(len(row["input_ids"]) for row in rows) > MODEL_MAX_LENGTH:
        raise ValueError(f"序列超过位置编码上限 {MODEL_MAX_LENGTH}")
    padded = principle_tokenizer.pad(
        rows,
        padding=True,
        return_attention_mask=False,
        return_tensors="pt",
    )
    result = padded["input_ids"]
    return result.to(device) if device is not None else result


def my_token_label(token_id):
    """把 ByT5 的单个字节 token 转成注意力图可读标签。"""
    token_id = int(token_id)
    if 3 <= token_id <= 258:
        byte_value = token_id - 3
        if byte_value == 0x20:
            return "␠"
        if 0x21 <= byte_value <= 0x7E:
            return chr(byte_value)
        return f"0x{byte_value:02X}"
    return str(principle_tokenizer.convert_ids_to_tokens(token_id))


def my_token_labels(text):
    """返回文本编码后每个 token 的可读标签。"""
    return [my_token_label(token_id) for token_id in my_encode(text)]


def my_decode(token_ids):
    """移除特殊 token，并把 token ID 解码为文本。"""
    return principle_tokenizer.decode(
        [int(token_id) for token_id in token_ids],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )


# 只保留 Encoder 和 Decoder 都能放入本章位置编码范围的句对。
translation_pairs = [
    (chinese, english)
    for chinese, english in RAW_PAIRS
    if len(my_encode(chinese, add_eos=True)) <= MODEL_MAX_LENGTH
    and len(my_encode(english)) + 1 <= MODEL_MAX_LENGTH
]
random.Random(SEED).shuffle(translation_pairs)
train_end = int(len(translation_pairs) * 0.8)
validation_end = int(len(translation_pairs) * 0.9)
TRAIN_EXAMPLES = translation_pairs[:train_end]
VALIDATION_EXAMPLES = translation_pairs[train_end:validation_end]
TEST_EXAMPLES = translation_pairs[validation_end:]

print("Tokenizer：", TOKENIZER_ID, "| 词表大小：", VOCAB_SIZE)
print("特殊 token：", SPECIAL_TOKEN_TO_ID)
print("训练/验证/测试样本：", len(TRAIN_EXAMPLES), len(VALIDATION_EXAMPLES), len(TEST_EXAMPLES))
print("样例 token：", TRAIN_EXAMPLES[0][0], "->", my_token_labels(TRAIN_EXAMPLES[0][0]))

### 2.3．Tokenizer 与模型的最小接口

- `VOCAB_SIZE` 决定源/目标 Embedding 与输出 LM Head 的词表维度。
- `PAD_ID` 用于 Padding Mask 和损失忽略位置，`BOS_ID` 启动 Decoder，`EOS_ID` 表示生成结束。
- `my_batch_encode()` 输出右侧补齐的 `[N,S]` token 张量；超过 `MODEL_MAX_LENGTH` 的输入会失败，因为位置编码没有对应位置。
- 本章原理模型与 5.3 节 MarianMT 使用不同 Tokenizer，二者的 token ID 和模型权重不可混用。

### 2.4．Transformer 数据流

源序列先经过 Encoder 形成“上下文记忆”；Decoder 一边读取已经生成的目标 token，一边查询这份记忆，最后给出下一个 token 在整个词表上的概率。

- **输入**：源 token `src [N,S]`，右移后的目标 token `tgt_in [N,T]`。
- **输出**：每个目标位置的未归一化词表分数 `logits [N,T,V]`。
- **训练目标**：第 $t$ 个输出预测真实的第 $t$ 个目标 token。

论文 Transformer 先把源序列编码成 Encoder Output，再由 Decoder 结合右移后的目标前缀产生目标词表分布。这里保持论文的 Post-LN、ReLU、6 层 Encoder 与 6 层 Decoder，作为后续原理实现和库迁移的共同基线：

<!-- diagram:transformer-overview -->

![架构图：Transformer 论文基线的数据流、Encoder memory 与 Decoder 条件生成](assets/figures/30_transformer/transformer-overview.svg)

[TikZ 源文件](assets/figures/30_transformer/transformer-overview.tex)


### 2.5．无上下文的逐位置预测基线

仅含 `Embedding → Linear` 的基线会独立处理每个 Token，因此同一 Token 在不同上下文中产生相同输出。该局限说明了引入 Attention 的必要性。

- **作用**：把 token 编号映射为向量，再映射成词表分数。
- **输入**：`tokens [N,S]`，元素为整数 token id。
- **中间结果**：`x [N,S,E]`。
- **输出**：`logits [N,S,V]`。

$$x_{b,s}=E[\mathrm{token}_{b,s}],\qquad \mathrm{logits}_{b,s}=x_{b,s}W+b$$


In [ ]:
# 先建立逐位置独立预测的基线，突出没有注意力时无法读取上下文。

class MyTokenWiseBaseline(nn.Module):
    """不含位置与注意力的逐 token 基线，用于证明上下文混合的必要性。"""
    def __init__(self, vocab_size: int, d_model: int):
        """创建词嵌入和逐位置词表投影，不引入任何 token 间信息交互。"""
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        """将 [N, S] token ID 映射为 [N, S, V] logits。"""
        x = self.embedding(tokens)       # [N,S] -> [N,S,E]
        return self.output(x)            # [N,S,E] -> [N,S,V]

# 8 维仅用于最小逐位置基线；增大容量仍不能弥补其缺少上下文读取机制。
baseline = MyTokenWiseBaseline(vocab_size=VOCAB_SIZE, d_model=8)
example_texts = ["我喜欢机器学习。", "他们正在学习机器学习。"]
example_tokens = my_batch_encode(example_texts)
baseline_logits = baseline(example_tokens)
for text in example_texts:
    print(f"{text} -> {my_token_labels(text)}")
print("编码形状:", tuple(example_tokens.shape))
print("logits :", tuple(baseline_logits.shape))
common_ids = set(my_encode(example_texts[0])) & set(my_encode(example_texts[1]))
shared_token_id = my_encode("机器学习")[0]  # 共享短语首个 byte token
if shared_token_id not in common_ids:
    raise RuntimeError("基线样例未包含预期的共享 token")
shared_positions = example_tokens.eq(shared_token_id).nonzero(as_tuple=False)
first, second = shared_positions[0], shared_positions[1]
print(f"两个句子共享的 ByT5 token：{my_token_label(shared_token_id)!r}")
print("不读取上下文时，同一个 token 的输出是否相同：",
      torch.allclose(baseline_logits[first[0], first[1]], baseline_logits[second[0], second[1]]))


#### 2.5.1．Embedding 的参数查找机制

词表可以看成一个矩阵 $W_{emb}\in\mathbb{R}^{V\times E}$。token id 是“行号”，取出的那一行就是该 token 的向量。反向传播会更新这些行。


In [ ]:
# 直接查看 Embedding 权重行，说明 token ID 如何索引可学习向量。

# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    table = baseline.embedding.weight.detach().cpu()
highlight_id = shared_token_id
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(table, cmap="Blues", aspect="auto")
ax.scatter(range(table.shape[1]), [highlight_id] * table.shape[1], facecolors="none",
           edgecolors="#E24A33", s=130, linewidths=2, label="共享 ByT5 token 的可学习向量")
ax.set(xlabel="特征维度 E", ylabel="token id（词表中的行）", title=r"Embedding 表 $W_{\mathrm{emb}} \in \mathbb{R}^{V \times E}$")
ax.legend(loc="upper right")
fig.colorbar(im, ax=ax, shrink=0.8, label="参数值")
plt.show()


### 2.6．Sinusoidal Positional Encoding

Self-Attention 本身对排列不敏感：如果只交换 token 的顺序，它看到的仍是一组相同向量。位置编码给每个位置一条独特、平滑的“坐标曲线”。

- **作用**：把顺序注入 token 表示。
- **输入**：token 向量 `x [N,S,E]`。
- **输出**：`x + PE [N,S,E]`，形状不变。

$$PE(pos,2i)=\sin\left(pos/10000^{2i/E}\right)$$
$$PE(pos,2i+1)=\cos\left(pos/10000^{2i/E}\right)$$

图像直觉：低维曲线振动快，负责区分邻近位置；高维曲线变化慢，负责描述更长距离。不同频率组合后，每个位置都获得近似唯一的坐标。

实现时将 `PE` 注册为 **persistent buffer**：它不参与梯度更新，但必须进入 `state_dict`。否则原理实现的训练过程可能保持正常，而 `save_pretrained()` / `from_pretrained()` 重载会将缺失的位置编码视为待初始化状态，导致同一权重产生不同 Logits。


In [ ]:
POSITION_ENCODING_BASE = 10_000.0  # 原始 Transformer 正弦编码的频率底数；修改会改变全部位置频率。
DEFAULT_POSITION_CACHE_LENGTH = 4096  # 演示缓存上限；只影响缓存占用，不扩展模型的长度泛化。

class MyPositionalEncoding(nn.Module):
    """生成并注入论文式正弦位置编码，输入输出形状均为 [N, S, E]。"""
    def __init__(self, d_model: int, max_len: int = DEFAULT_POSITION_CACHE_LENGTH, dropout: float = 0.0):
        """预计算长度上限内的正弦位置编码，并将其注册为非参数缓冲区。"""
        super().__init__()
        position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)      # [L,1]
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(POSITION_ENCODING_BASE) / d_model)
        )                                                                       # [ceil(E/2)]
        pe = torch.zeros(max_len, d_model)                                      # [L,E]
        pe[:, 0::2] = torch.sin(position * div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position * div_term[: pe[:, 1::2].shape[1]])
        # 纳入 state_dict：`save_pretrained()` / `from_pretrained()` 必须精确恢复位置编码。
        self.register_buffer("pe", pe.unsqueeze(0), persistent=True)        # [1,L,E]
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """向 [N, S, E] 隐状态加入对应位置编码并施加 dropout。序列超长时抛出 ValueError。"""
        if x.size(1) > self.pe.size(1):
            raise ValueError(f"序列长度 {x.size(1)} 超过 max_len={self.pe.size(1)}")
        return self.dropout(x + self.pe[:, : x.size(1)].to(dtype=x.dtype))

pe_demo = MyPositionalEncoding(d_model=32, max_len=80)
pe = pe_demo.pe[0].cpu()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(pe.T, cmap="RdBu", aspect="auto", vmin=-1, vmax=1)
axes[0].set(xlabel="位置 pos", ylabel="维度 i", title="位置编码热力图：每一列是一个位置坐标")
for dim in [0, 2, 8, 16, 30]:
    axes[1].plot(pe[:, dim], label=f"维度 {dim}")
axes[1].set(xlabel="位置 pos", ylabel="PE 值", title="不同维度 = 不同频率的时钟")
axes[1].legend(ncol=2)
plt.tight_layout()
plt.show()


### 2.7．Scaled Dot-Product Attention

每个查询位置通过三种角色组织输入：

- **Query（Q）**：查询位置希望匹配的特征。
- **Key（K）**：候选位置用于参与匹配的特征。
- **Value（V）**：匹配后参与加权聚合的信息。

- **输入**：`Q [N,H,L,Eh]`、`K [N,H,S,Eh]`、`V [N,H,S,Eh]`。
- **输出**：上下文 `context [N,H,L,Eh]` 和权重 `weights [N,H,L,S]`。

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{E_h}}\right)V$$

图解公式分三步：$QK^{\top}$ 计算相似度 → $\operatorname{softmax}$ 变成每行和为 $1$ 的权重 → 权重乘 $V$ 做加权求和。除以 $\sqrt{E_h}$ 是为了避免维度大时点积过大、softmax 过早饱和。

下一单元的 `F.one_hot` 仅构造一组可确定复现、彼此正交的可视化特征，用于隔离观察“匹配—权重—聚合”机制。它不是 Transformer 的真实 Token 输入路径；真实路径是 Token ID 经 Embedding 变为稠密向量。One-hot 的基础契约及其与 Embedding 的等价关系见 [10_foundations.ipynb](10_foundations.ipynb)。


In [ ]:
def my_scaled_dot_product_attention(
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attn_mask: torch.Tensor | None = None,
    dropout: nn.Module | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """对齐 nn.Transformer：布尔 attn_mask 中 True=屏蔽。"""
    scores = query @ key.transpose(-2, -1) / math.sqrt(query.size(-1))        # [N,H,L,S]
    if attn_mask is not None:
        scores = scores.masked_fill(attn_mask, float("-inf"))
    weights = torch.softmax(scores, dim=-1)                                  # [N,H,L,S]
    if dropout is not None:
        weights = dropout(weights)
    context = weights @ value                                                # [N,H,L,Eh]
    return context, weights

# 使用短文本的官方 ByT5 byte token 图解公式；One-hot 仅是隔离 Attention 机制的正交特征夹具。
attention_text = "注意"
attention_tokens = my_token_labels(attention_text)
visual_features = F.one_hot(
    torch.arange(len(attention_tokens)), num_classes=len(attention_tokens)
).float()
q = k = v = visual_features[None, None, :, :]
context, weights = my_scaled_dot_product_attention(q, k, v)
scores = (q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))).squeeze().numpy()
weights_np = weights.squeeze().numpy()
context_np = context.squeeze().numpy()

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, matrix, title in zip(
    axes, [scores, weights_np, context_np],
    [r"① 相似度 $QK^{\top} / \sqrt{E_h}$", "② softmax 后的读取权重", r"③ 权重 $\times V$ 得到上下文"]
):
    im = ax.imshow(matrix, cmap="Blues", aspect="auto")
    # 直接按矩阵尺寸标注数值。
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xticks(range(len(attention_tokens)), attention_tokens, rotation=45)
    ax.set_yticks(range(len(attention_tokens)), attention_tokens)
    fig.colorbar(im, ax=ax, shrink=0.7)
plt.tight_layout()
plt.show()

row_sum_error = float((weights.sum(dim=-1) - 1.0).abs().max())
print(f"Attention 每行概率和相对 1 的最大误差：{row_sum_error:.2e}")


#### 2.7.1．Attention：从匹配到加权重心

**学习问题**：固定一个 Query 后，匹配分数如何变成读取概率，这些概率又如何把多个 Value 合成为一个 Context？

下面的确定性静态分镜直接复用上一单元的 `q`、`k`、`v`、`scores`、`weights` 与 `context`，并固定一个 Query 保持观察对象不变：

1. **匹配**：Query 与每个 Key 形成一条候选读取关系，边上标注缩放点积分数。
2. **归一化**：Softmax 把同一行分数变成非负且和为 $1$ 的读取权重。
3. **搬运信息**：每个权重缩放对应的 Value；彩色向量首尾相接，终点等于投影后的 Context。
4. **加权重心**：Context 位于 Value 的加权组合位置，连线粗细继续沿用同一组权重。

绘图前验证四个可证伪条件：权重非负、每行权重和为 $1$、`context == weights @ v`，以及固定正弦/余弦二维线性投影中 `P(context) == weights @ P(v)`。二维投影仅用于呈现高维对象的几何关系；回到原空间的重构均方根误差会作为信息损失摘要报告，不应预期为零。Attention 权重只表示当前层、当前头、当前输入下的读取比例，二维位置也会随投影方法改变；两者都不能单独推出特征重要性、因果归因或模型行为原因。


In [ ]:
# 直接复用上一单元的 Attention 中间张量，先验证数值契约，再绘制固定 Query 的几何分镜。
import numpy as np

ATTENTION_STORY_ATOL = 1e-6
query_story = q.detach().float().cpu()[0, 0]
key_story = k.detach().float().cpu()[0, 0]
value_story = v.detach().float().cpu()[0, 0]
weight_story = weights.detach().float().cpu()[0, 0]
context_story = context.detach().float().cpu()[0, 0]
score_story = np.asarray(scores, dtype=np.float64)

minimum_weight = float(weight_story.min())
row_sum_max_error = float((weight_story.sum(dim=-1) - 1.0).abs().max())
weighted_context_story = weight_story @ value_story
context_max_error = float((context_story - weighted_context_story).abs().max())
if minimum_weight < -ATTENTION_STORY_ATOL:
    raise AssertionError(f"Attention 权重出现负值：{minimum_weight:.3e}")
if row_sum_max_error > ATTENTION_STORY_ATOL:
    raise AssertionError(f"Attention 权重行和误差过大：{row_sum_max_error:.3e}")
if context_max_error > ATTENTION_STORY_ATOL:
    raise AssertionError(f"context 与 weights @ V 不一致：{context_max_error:.3e}")

query_np = query_story.numpy().astype(np.float64, copy=False)
key_np = key_story.numpy().astype(np.float64, copy=False)
value_np = value_story.numpy().astype(np.float64, copy=False)
weight_np = weight_story.numpy().astype(np.float64, copy=False)
context_story_np = context_story.numpy().astype(np.float64, copy=False)

# 所有角色共享固定的低频正弦/余弦正交基；同一特征维始终映射到同一二维方向。
projection_source = np.concatenate([query_np, key_np, value_np, context_story_np], axis=0)
projection_center = projection_source.mean(axis=0, keepdims=True)
projection_feature_count = projection_source.shape[1]
if projection_feature_count < 3:
    raise ValueError("Attention 几何分镜至少需要三个特征维度")
projection_angles = 2.0 * np.pi * np.arange(projection_feature_count) / projection_feature_count
projection_basis = np.stack([np.cos(projection_angles), np.sin(projection_angles)], axis=1)
projection_basis /= np.linalg.norm(projection_basis, axis=0, keepdims=True)
projection_basis_error = float(
    np.max(np.abs(projection_basis.T @ projection_basis - np.eye(2)))
)
if projection_basis_error > ATTENTION_STORY_ATOL:
    raise AssertionError(f"二维投影基不满足正交契约：{projection_basis_error:.3e}")

def my_attention_story_project(points: np.ndarray) -> np.ndarray:
    """使用本单元冻结的二维基，把 Attention 角色投影到同一坐标系。"""
    return (points - projection_center) @ projection_basis

query_points_2d = my_attention_story_project(query_np)
key_points_2d = my_attention_story_project(key_np)
value_points_2d = my_attention_story_project(value_np)
context_points_2d = my_attention_story_project(context_story_np)
projected_context_from_values = weight_np @ value_points_2d
projected_context_max_error = float(
    np.max(np.abs(context_points_2d - projected_context_from_values))
)
if projected_context_max_error > ATTENTION_STORY_ATOL:
    raise AssertionError(
        f"二维投影未保持 Attention 加权关系：{projected_context_max_error:.3e}"
    )
context_reconstructed_from_2d = context_points_2d @ projection_basis.T + projection_center
projection_reconstruction_rmse = float(
    np.sqrt(np.mean((context_story_np - context_reconstructed_from_2d) ** 2))
)
if not np.isfinite(projection_reconstruction_rmse):
    raise AssertionError("二维投影重构误差不是有限值")

focus_query_index = min(1, query_story.shape[0] - 1)
focus_scores = score_story[focus_query_index]
focus_weights = weight_np[focus_query_index]
focus_context_2d = context_points_2d[focus_query_index]
focus_projected_context = projected_context_from_values[focus_query_index]
focus_contributions = focus_weights[:, None] * value_points_2d
focus_cumulative_path = np.vstack([
    np.zeros(2, dtype=np.float64), np.cumsum(focus_contributions, axis=0)
])
top_key_index = int(np.argmax(focus_weights))

story_background = "#0B1020"
story_foreground = "#E6EDF7"
story_muted = "#93A4B8"
story_query_color = "#FFD166"
story_context_color = "#F72585"
story_palette = ["#4CC9F0", "#F9844A", "#90BE6D", "#A78BFA", "#F9C74F", "#43AA8B"]
token_colors = [story_palette[index % len(story_palette)] for index in range(len(attention_tokens))]

geometry_for_limits = np.vstack([
    query_points_2d, key_points_2d, value_points_2d, context_points_2d, focus_cumulative_path
])
x_min, y_min = geometry_for_limits.min(axis=0)
x_max, y_max = geometry_for_limits.max(axis=0)
x_margin = max(0.25, 0.18 * max(x_max - x_min, 1.0))
y_margin = max(0.25, 0.18 * max(y_max - y_min, 1.0))

def my_style_attention_geometry_axis(axis):
    """统一三幅几何分镜的坐标范围与深色叙事样式。"""
    axis.set_facecolor(story_background)
    axis.axhline(0.0, color=story_muted, linewidth=0.7, alpha=0.35)
    axis.axvline(0.0, color=story_muted, linewidth=0.7, alpha=0.35)
    axis.set_xlim(x_min - x_margin, x_max + x_margin)
    axis.set_ylim(y_min - y_margin, y_max + y_margin)
    axis.set_aspect("equal", adjustable="box")
    axis.set_xlabel("投影维 1", color=story_muted)
    axis.set_ylabel("投影维 2", color=story_muted)
    axis.tick_params(colors=story_muted, labelsize=8)
    for spine in axis.spines.values():
        spine.set_color("#27324A")

fig, story_axes = plt.subplots(1, 4, figsize=(18, 4.8), facecolor=story_background)
for story_axis in story_axes:
    story_axis.set_facecolor(story_background)

# ① 同一个 Query 与全部 Key 建立匹配关系；颜色在后续分镜中保持不变。
match_axis = story_axes[0]
focus_query_xy = query_points_2d[focus_query_index]
score_span = float(np.ptp(focus_scores))
score_strength = (focus_scores - focus_scores.min()) / (score_span if score_span > 0 else 1.0)
for key_index, key_xy in enumerate(key_points_2d):
    match_axis.plot(
        [focus_query_xy[0], key_xy[0]], [focus_query_xy[1], key_xy[1]],
        color=token_colors[key_index], linewidth=1.0 + 2.5 * score_strength[key_index],
        alpha=0.28 + 0.62 * score_strength[key_index], zorder=1
    )
    midpoint = (focus_query_xy + key_xy) / 2
    match_axis.text(
        midpoint[0], midpoint[1], f"{focus_scores[key_index]:.2f}",
        color=story_foreground, fontsize=7, ha="center", va="center"
    )
    match_axis.scatter(*key_xy, s=90, color=token_colors[key_index], edgecolor=story_background, zorder=3)
    match_axis.annotate(
        attention_tokens[key_index], key_xy, xytext=(5, 5), textcoords="offset points",
        color=token_colors[key_index], fontsize=8
    )
match_axis.scatter(
    *focus_query_xy, s=260, marker="*", facecolors="none",
    edgecolors=story_query_color, linewidths=1.8, zorder=4
)
match_axis.set_title("① 匹配：Query 对全部 Key 打分", color=story_foreground, pad=12)
my_style_attention_geometry_axis(match_axis)

# ② 读取概率沿用 Key/Value 的对象颜色。
weight_axis = story_axes[1]
weight_positions = np.arange(len(attention_tokens))
weight_axis.barh(weight_positions, focus_weights, color=token_colors, alpha=0.9)
for key_index, weight_value in enumerate(focus_weights):
    weight_axis.text(
        weight_value + max(focus_weights.max() * 0.025, 0.003), key_index, f"{weight_value:.3f}",
        color=story_foreground, fontsize=8, va="center"
    )
weight_axis.set_yticks(weight_positions, attention_tokens)
weight_axis.invert_yaxis()
weight_axis.set_xlim(0.0, max(0.25, float(focus_weights.max()) * 1.28))
weight_axis.set_xlabel("读取权重", color=story_muted)
weight_axis.set_title("② Softmax：权重非负且和为 1", color=story_foreground, pad=12)
weight_axis.tick_params(colors=story_muted, labelsize=8)
for spine in weight_axis.spines.values():
    spine.set_color("#27324A")

# ③ 每段箭头都是 w_j × P(V_j)，首尾相接的终点应与 P(context) 重合。
contribution_axis = story_axes[2]
for value_index in range(len(attention_tokens)):
    start_xy = focus_cumulative_path[value_index]
    end_xy = focus_cumulative_path[value_index + 1]
    contribution_axis.annotate(
        "", xy=end_xy, xytext=start_xy,
        arrowprops=dict(
            arrowstyle="-|>", color=token_colors[value_index],
            linewidth=1.4 + 4.0 * focus_weights[value_index], shrinkA=0, shrinkB=0
        )
    )
    contribution_axis.text(
        *((start_xy + end_xy) / 2), attention_tokens[value_index],
        color=token_colors[value_index], fontsize=8, ha="center", va="bottom"
    )
contribution_axis.scatter(0.0, 0.0, s=35, color=story_muted, zorder=3)
contribution_axis.scatter(
    *focus_projected_context, s=150, marker="D", color=story_context_color,
    edgecolor=story_foreground, linewidth=1.0, zorder=4
)
contribution_axis.set_title("③ 搬运：缩放后的 Value 首尾相加", color=story_foreground, pad=12)
my_style_attention_geometry_axis(contribution_axis)

# ④ Context 是所有 Value 的加权重心；连线粗细继续编码同一份权重。
barycenter_axis = story_axes[3]
for value_index, value_xy in enumerate(value_points_2d):
    barycenter_axis.plot(
        [value_xy[0], focus_context_2d[0]], [value_xy[1], focus_context_2d[1]],
        color=token_colors[value_index], linewidth=0.8 + 5.0 * focus_weights[value_index],
        alpha=0.3 + 0.6 * focus_weights[value_index] / max(float(focus_weights.max()), 1e-12)
    )
    barycenter_axis.scatter(*value_xy, s=95, color=token_colors[value_index], edgecolor=story_background, zorder=3)
    barycenter_axis.annotate(
        attention_tokens[value_index], value_xy, xytext=(5, 5), textcoords="offset points",
        color=token_colors[value_index], fontsize=8
    )
barycenter_axis.scatter(
    *focus_context_2d, s=190, marker="D", color=story_context_color,
    edgecolor=story_foreground, linewidth=1.2, zorder=4, label="Context"
)
barycenter_axis.annotate(
    "Context", focus_context_2d, xytext=(7, -13), textcoords="offset points",
    color=story_context_color, fontsize=9, fontweight="bold"
)
barycenter_axis.set_title("④ 聚合：Context 是 Value 的加权重心", color=story_foreground, pad=12)
my_style_attention_geometry_axis(barycenter_axis)

fig.suptitle(
    f"Attention：从匹配到加权重心｜固定 Query={attention_tokens[focus_query_index]}",
    color=story_foreground, fontsize=15, y=0.99
)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.92))
plt.show()

print("Attention 几何分镜数值摘要")
print(f"  固定 Query：{attention_tokens[focus_query_index]}")
print(
    f"  最高权重 Key：{attention_tokens[top_key_index]} "
    f"(score={focus_scores[top_key_index]:.3f}, weight={focus_weights[top_key_index]:.3f})"
)
print(f"  权重最小值：{minimum_weight:.3e}")
print(f"  权重行和最大误差：{row_sum_max_error:.3e}")
print(f"  context = weights @ V 最大绝对误差：{context_max_error:.3e}")
print(f"  二维空间加权重构最大误差：{projected_context_max_error:.3e}")
print(f"  二维投影回原空间 RMSE（信息损失）：{projection_reconstruction_rmse:.3e}")


图中三幅矩阵必须满足同一条计算链：相似度经过 Softmax 后每行概率和为 1，随后同一行权重对 $V$ 做加权求和。若行和误差超出浮点容差，说明归一化维度或张量形状存在错误。该图用于验证信息读取的数值契约；Attention 权重较高只表示当前模型在这一层、这一头中的读取比例，不能单独作为特征重要性或因果解释。

### 2.8．Mask 的可见性契约

Transformer 需要两类布尔 mask。为与 `nn.Transformer` 保持一致，本文统一规定 **`True` 表示屏蔽，`False` 表示允许读取**：

1. **Padding mask**：真实 token 可见，`<pad>` 不可见，避免补齐符污染上下文。
2. **Causal mask**：Decoder 的位置 $t$ 只能看 $\le t$ 的位置，避免训练时偷看答案。

- `key_padding_mask`：`[N,S]`，与官方接口相同，在注意力内部扩展到各个头。
- `attn_mask`：`[L,S]`；Decoder causal mask 是 `[T,T]`，严格上三角为 `True`。
- 同时存在时，二者逻辑或：任意一个要求屏蔽，该位置就不可见。


In [ ]:
# 分别构造 padding mask 与 causal mask，控制批次补齐和未来信息可见性。

def my_make_key_padding_mask(tokens: torch.Tensor, pad_id: int = PAD_ID) -> torch.Tensor:
    """根据 PAD ID 生成形状为 [N, S] 的布尔掩码，其中 True 表示不可见。"""
    return tokens.eq(pad_id)                                                    # [N,S]，True=屏蔽

def my_make_causal_mask(length: int, device: torch.device) -> torch.Tensor:
    """生成形状为 [T, T] 的上三角因果掩码，其中 True 屏蔽未来位置。"""
    return torch.ones(length, length, dtype=torch.bool, device=device).triu(diagonal=1)  # [T,T]

# 构造 mask，显式控制哪些 token 位置可以参与后续计算。
mask_texts = ["我爱学习。", "注意力读取上下文。"]
tokens = my_batch_encode(mask_texts)
print("较短句子：", mask_texts[0], "->", my_token_labels(mask_texts[0]))
print("较长句子：", mask_texts[1], "->", my_token_labels(mask_texts[1]))
padding = my_make_key_padding_mask(tokens)                                        # [N,S]
causal = my_make_causal_mask(tokens.size(1), tokens.device)                       # [S,S]
combined = padding[:, None, None, :] | causal[None, None, :, :]                # [N,1,S,S]

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
items = [(padding[0].unsqueeze(0).float(), "Key padding mask"),
         (causal.float(), "Causal attn_mask"),
         (combined[0, 0].float(), "两者逻辑或")]
for ax, (matrix, title) in zip(axes, items):
    ax.imshow(matrix, cmap="Greens", vmin=0, vmax=1, aspect="auto")
    ax.set(title=title + "（1=屏蔽）", xlabel="Key 位置（被读取）", ylabel="Query 位置（发起读取）")
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_yticks(range(matrix.shape[0]))
plt.tight_layout()
plt.show()


### 2.9．Causal / Non-causal 与三类 Transformer 架构

这里要分开两个容易混在一起的维度：

1. **Causal / Non-causal** 描述一次 Attention 的**可见性规则**，由 Attention Mask 决定。
2. **Encoder-only / Decoder-only / Encoder–Decoder** 描述模型的**模块组成与信息流**。

**Causal Attention（因果注意力）**规定目标位置 $t$ 只能读取位置 $\le t$ 的 Key/Value，不能读取未来 Token。它使用严格上三角 Mask，保证 Teacher Forcing 训练时虽然整条目标序列同时存在于张量中，当前位置仍然不能偷看答案。这里的 Causal 是“按生成顺序限制信息流”，不是统计学中的因果推断。

**Non-causal Attention（非因果注意力，也常称双向或全可见注意力）**允许每个 Query 读取同一序列中的全部有效位置，因此一个 Token 可以同时利用左右上下文。Non-causal 不等于完全没有 Mask：Padding、文档边界或业务规则仍可能屏蔽部分位置。

<!-- diagram:transformer-family-map -->

![架构图：三类 Transformer 架构的注意力可见性与信息流对照](assets/figures/30_transformer/transformer-family-map.svg)

[TikZ 源文件](assets/figures/30_transformer/transformer-family-map.tex)

| 典型架构 | Self-Attention 的默认可见性 | Cross-Attention | 典型输出与任务 | 代表模型 |
|---|---|---|---|---|
| Encoder-only | Non-causal：读取左右上下文 | 无 | 每个位置的上下文表示；分类、标注、检索 | BERT |
| Decoder-only | Causal：只读取当前与过去 | 无 | 下一个 Token 的 logits；自回归生成 | GPT、Llama、Qwen |
| Encoder–Decoder | Encoder Non-causal；Decoder Causal | 有 | 条件生成；翻译、摘要等 Sequence-to-Sequence 任务 | 原始 Transformer、T5、BART、Marian |

在标准 Encoder–Decoder 中，Decoder 的目标 Self-Attention 是 Causal；但 Cross-Attention 通常**不对源序列使用三角因果 Mask**。某个目标位置只能基于已经生成的目标前缀发起 Query，却可以读取 Encoder 产生的全部有效 Source Memory；源端 `<pad>` 仍由 `memory_key_padding_mask` 屏蔽。

因此不能把两组术语简单画等号：

- `Causal LM` 通常采用 Decoder-only 架构，但 Encoder–Decoder 的 Decoder 同样使用 Causal Self-Attention。
- Decoder-only Block 不是把原始 Transformer Decoder 原样保留下来；它通常移除了 Encoder 和 Cross-Attention，只保留带 Causal Mask 的 Self-Attention 与 FFN 主干。
- BERT 是 Encoder-only，而不是 Encoder–Decoder；Llama、GPT、Qwen 是 Decoder-only，而不是 Encoder–Decoder。
- 架构名称描述典型设计，不是不可改变的物理定律；特殊训练目标可以使用自定义可见性 Mask，所以实现与评审时仍要检查每个 Attention 子层的实际 Mask。


<!-- theory-math-contract:v1 -->
### 2.10．核心机制的语言与数学表达

Attention 先用查询与键计算相关性，再把归一化权重用于值向量的加权聚合。缩放项抑制维度增大造成的点积方差：

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V,\qquad
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V
$$

其中，$X\in\mathbb{R}^{B\times L\times d_{\mathrm{model}}}$，$Q,K\in\mathbb{R}^{B\times H\times L\times d_k}$，$V\in\mathbb{R}^{B\times H\times L\times d_v}$，$M$ 是可广播的 Mask。标准多头实现通常要求 $d_{\mathrm{model}}\bmod H=0$，从而每头维度 $d_k=d_{\mathrm{model}}/H$。`my_scaled_dot_product_attention` 对应公式主体，`torch.nn.functional.scaled_dot_product_attention` 对应生产接口；FlashAttention 等 Kernel 改变访存与数值路径，不改变上述数学语义。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．多头注意力

一个头只能在一个投影空间里计算相似度。多头注意力先把 $E$ 维表示投影并拆成 $H$ 份，每个头独立做注意力，再拼接并通过输出投影融合。不同头可以分别学习语法、指代、局部关系或长距离关系。

- **输入**：`query [N,L,E]`、`key/value [N,S,E]`。
- **拆头后**：Query 为 `[N,H,L,Eh]`，Key/Value 为 `[N,H,S,Eh]`。
- **输出**：`output [N,L,E]`，形状回到模型维度；另返回 `weights [N,H,L,S]` 便于解释。

$$\mathrm{head}_i=\mathrm{Attention}(QW_i^Q,KW_i^K,VW_i^V)$$
$$\mathrm{MHA}(Q,K,V)=\mathrm{Concat}(\mathrm{head}_1,\ldots,\mathrm{head}_H)W^O$$


In [ ]:
# 从零实现多头注意力的投影、分头、缩放点积和合并流程。

class MyMultiheadAttention(nn.Module):
    """从零实现多头注意力的投影、分头、掩码、聚合与输出映射。"""
    def __init__(self, d_model: int, nhead: int, dropout: float = 0.0):
        """校验头维度并创建 Q、K、V 与输出线性投影。"""
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError(f"d_model={d_model} 必须能被 nhead={nhead} 整除")
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)

    def my_split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """将 [N, L, E] 隐状态重排为 [N, H, L, Eh] 多头表示。"""
        batch, length, _ = x.shape
        return x.view(batch, length, self.nhead, self.head_dim).transpose(1, 2)

    def my_merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """将 [N, H, L, Eh] 多头表示合并回 [N, L, E]。"""
        batch, _, length, _ = x.shape
        return x.transpose(1, 2).contiguous().view(batch, length, self.d_model)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        """执行带注意力掩码和 Padding 掩码的多头注意力，返回输出与各头权重。"""
        q = self.my_split_heads(self.q_proj(query))                              # [N,H,L,Eh]
        k = self.my_split_heads(self.k_proj(key))                                # [N,H,S,Eh]
        v = self.my_split_heads(self.v_proj(value))                              # [N,H,S,Eh]
        combined_mask = None
        if attn_mask is not None:
            if attn_mask.dtype != torch.bool or attn_mask.dim() != 2:
                raise ValueError("渐进版 attn_mask 简化为二维 bool 张量 [Q,K]")
            combined_mask = attn_mask[None, None, :, :]
        if key_padding_mask is not None:
            if key_padding_mask.dtype != torch.bool or key_padding_mask.dim() != 2:
                raise ValueError("key_padding_mask 必须是二维 bool 张量 [N,S]")
            padding_mask = key_padding_mask[:, None, None, :]
            combined_mask = padding_mask if combined_mask is None else combined_mask | padding_mask
        context, weights = my_scaled_dot_product_attention(
            q, k, v, attn_mask=combined_mask,
            dropout=self.attn_dropout if self.training else None
        )
        output = self.out_proj(self.my_merge_heads(context))                    # [N,L,E]
        return output, weights

# 16 维与 4 个头形成每头 4 维的 CPU 数值检查；调整后必须保持整除。
mha = MyMultiheadAttention(d_model=16, nhead=4)
mha_texts = ["注意力读取上下文。", "attention reads context."]
mha_token_ids = my_batch_encode(mha_texts)
mha_embedding = nn.Embedding(VOCAB_SIZE, 16)
x = mha_embedding(mha_token_ids)
mha_output, mha_weights = mha(
    x, x, x, key_padding_mask=my_make_key_padding_mask(mha_token_ids)
)
for text in mha_texts:
    print(f"{text} -> {my_token_labels(text)}")
print("输入     :", tuple(x.shape))
print("输出     :", tuple(mha_output.shape))
print("注意力权重:", tuple(mha_weights.shape))


#### 3.1.1．逐头读取模式的诊断可视化

学习问题是：多个 Attention Head 是否在各自的投影子空间中形成独立的读取矩阵。下图将同一个短序列送入上一单元的 `MyMultiheadAttention`，完整保留 `[N,H,L,S]` 中的 Head 维度。验收条件是每个 Head 的每个 Query 行概率和为 1。


In [ ]:
# 使用同一原理模块的真实逐头权重，比较各投影子空间中的读取比例。
visual_mha_text = "注意"
visual_mha_ids = my_batch_encode([visual_mha_text])
visual_mha_x = mha_embedding(visual_mha_ids)
_, visual_mha_weights = mha(visual_mha_x, visual_mha_x, visual_mha_x)
visual_mha_weights = visual_mha_weights[0].detach().float().cpu()  # [H,L,S]
visual_mha_labels = my_token_labels(visual_mha_text)
expected_visual_shape = (mha.nhead, len(visual_mha_labels), len(visual_mha_labels))
if tuple(visual_mha_weights.shape) != expected_visual_shape:
    raise RuntimeError(
        f"逐头 Attention 形状不符合 [H,L,S] 契约：expected={expected_visual_shape}, actual={tuple(visual_mha_weights.shape)}"
    )
row_sum_error = float((visual_mha_weights.sum(dim=-1) - 1.0).abs().max())
if row_sum_error > 1e-5:
    raise RuntimeError(f"逐头 Attention 行和不满足概率契约：{row_sum_error:.2e}")

fig, axes_grid = plt.subplots(
    1, visual_mha_weights.size(0), figsize=(16, 3.8),
    sharex=True, sharey=True, squeeze=False,
)
axes = list(axes_grid[0])
for head_index, ax in enumerate(axes):
    image = ax.imshow(visual_mha_weights[head_index], cmap="viridis", vmin=0.0, vmax=1.0)
    ax.set_title(f"Head {head_index}")
    ax.set_xticks(range(len(visual_mha_labels)), visual_mha_labels, rotation=45)
    ax.set_yticks(range(len(visual_mha_labels)), visual_mha_labels)
    ax.set_xlabel("Key 位置")
axes[0].set_ylabel("Query 位置")
fig.colorbar(image, ax=axes, shrink=0.75, label="读取概率")
fig.suptitle("同一序列在不同 Attention Head 中的读取矩阵")
plt.show()
print({"weights_shape": tuple(visual_mha_weights.shape), "max_row_sum_error": row_sum_error})


不同 Head 可以呈现不同的权重分布，因为它们使用不同的 $W_Q$、$W_K$ 和 $W_V$。本单元仍处于随机初始化阶段，因此图形只证明 Head 维度和概率契约确实独立存在，不能据此声称某个 Head 已经学到语法、指代或其他稳定语义。即使训练后出现清晰模式，Attention 权重也不是严格的因果归因。


#### 3.1.2．从三个独立投影到一个打包投影

Q、K、V 在数学语义上始终是三种角色，并分别拥有独立参数。原理实现使用三个 `Linear`，直接对应三个公式：

$$Q=XW_Q^{\top}+b_Q,\qquad K=XW_K^{\top}+b_K,\qquad V=XW_V^{\top}+b_V$$

在 Self-Attention 中，三个投影读取同一个输入 $X$。因此可以沿输出维拼接三组权重和偏置，把三次形式相同的线性投影写成一次更大的线性投影：

$$W_{QKV}=\operatorname{Concat}_0(W_Q,W_K,W_V)\in\mathbb{R}^{3E\times E}$$
$$b_{QKV}=\operatorname{Concat}_0(b_Q,b_K,b_V)\in\mathbb{R}^{3E}$$
$$QKV=\operatorname{Linear}(X,W_{QKV},b_{QKV})\in\mathbb{R}^{N\times L\times 3E}$$

最后沿特征维把 `QKV` 等分为三个 `[N,L,E]` 张量。该变换只改变参数布局和执行方式：$W_Q$、$W_K$、$W_V$ 仍占据互不重叠的参数区间，参数量仍为 $3E^2+3E$，没有发生权重共享。

| 层次 | 表达 | 张量变化 |
|---|---|---|
| 原理表达 | `q_proj`、`k_proj`、`v_proj` | 三次 `[N,L,E] → [N,L,E]` |
| 打包表达 | `qkv_proj` | 一次 `[N,L,E] → [N,L,3E]` |
| PyTorch 参数布局 | `in_proj_weight`、`in_proj_bias` | `[3E,E]`、`[3E]`，输出后按 Q/K/V 切分 |

Cross-Attention 的 Query 与 Key/Value 来自不同张量，不能直接使用一次 `X → QKV` 计算；实现可以继续打包存储参数，但执行时需要分别投影 Query 和 Key/Value。


In [ ]:
# 拼接三个独立投影的参数，验证一次 QKV 投影与三次投影数值等价。
with torch.no_grad():
    separate_q = mha.q_proj(x)
    separate_k = mha.k_proj(x)
    separate_v = mha.v_proj(x)

    packed_weight = torch.cat(
        (mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight), dim=0
    )                                                                          # [3E,E]
    packed_bias = torch.cat(
        (mha.q_proj.bias, mha.k_proj.bias, mha.v_proj.bias), dim=0
    )                                                                          # [3E]
    packed_qkv = F.linear(x, packed_weight, packed_bias)                        # [N,L,3E]
    packed_q, packed_k, packed_v = packed_qkv.chunk(3, dim=-1)

    packing_error = torch.stack([
        (separate_q - packed_q).abs().max(),
        (separate_k - packed_k).abs().max(),
        (separate_v - packed_v).abs().max(),
    ]).max()

print("三个独立输出：", tuple(separate_q.shape), tuple(separate_k.shape), tuple(separate_v.shape))
print("一次打包输出：", tuple(packed_qkv.shape))
print("拆分后的输出：", tuple(packed_q.shape), tuple(packed_k.shape), tuple(packed_v.shape))
print(f"打包前后最大误差：{float(packing_error):.2e}")


#### 3.1.3．MHA 与 `nn.MultiheadAttention` 的数值对照

一致的是计算契约，简化的是参数存储和高性能执行路径：

| 步骤 | 渐进版 | PyTorch 库 |
|---|---|---|
| Q/K/V 投影 | 3 个独立 `Linear` | 通常用一个打包的 `in_proj_weight` |
| 拆头 | `[N,L,E] → [N,H,L,Eh]` | 相同 |
| 分数 | $QK^{\top} / \sqrt{E_h}$ | 相同 |
| Mask | bool `True=屏蔽`，padding 与 attn mask 合并 | 相同外部语义 |
| 归一化 | 最后一维 softmax | 相同 |
| Dropout | 作用在注意力权重上 | 相同 |
| 聚合 | `weights @ V` | 相同 |
| 合并与输出 | 拼头后乘 `W^O` | 相同 |

验证时将库模块打包的 Q/K/V 参数拆分并复制到原理实现，同时比较输出与逐头权重。数值对齐后，后续 Encoder 与 Decoder 使用 `nn.MultiheadAttention(batch_first=True)`。无需注意力图时设置 `need_weights=False`，PyTorch 可在条件满足时选择优化的 SDPA 实现；Cross-Attention 可视化路径则显式请求逐头权重。


In [ ]:
# 复制标准层参数后并排计算，验证原理 MHA 与库实现的数值差异。

with torch.random.fork_rng():
    torch.manual_seed(SEED)
    simple_mha = MyMultiheadAttention(d_model=16, nhead=4, dropout=0.0).eval()
    # 固定形状：library_mha.in_proj_weight.shape = [48, 16]；library_mha.out_proj.weight.shape = [16, 16]。
    library_mha = nn.MultiheadAttention(
        embed_dim=16, num_heads=4, dropout=0.0, batch_first=True
    ).eval()

    # 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
    with torch.no_grad():
        q_weight, k_weight, v_weight = library_mha.in_proj_weight.chunk(3, dim=0)
        q_bias, k_bias, v_bias = library_mha.in_proj_bias.chunk(3, dim=0)
        simple_mha.q_proj.weight.copy_(q_weight)
        simple_mha.k_proj.weight.copy_(k_weight)
        simple_mha.v_proj.weight.copy_(v_weight)
        simple_mha.q_proj.bias.copy_(q_bias)
        simple_mha.k_proj.bias.copy_(k_bias)
        simple_mha.v_proj.bias.copy_(v_bias)
        simple_mha.out_proj.load_state_dict(library_mha.out_proj.state_dict())

    comparison_texts = ["thank you.", "attention reads context."]
    comparison_ids = my_batch_encode(comparison_texts)
    comparison_embedding = nn.Embedding(VOCAB_SIZE, 16)
    mha_input = comparison_embedding(comparison_ids)
    mha_causal_mask = my_make_causal_mask(mha_input.size(1), mha_input.device)
    mha_padding_mask = my_make_key_padding_mask(comparison_ids)
    # 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
    with torch.no_grad():
        simple_output, simple_weights = simple_mha(
            mha_input, mha_input, mha_input,
            attn_mask=mha_causal_mask, key_padding_mask=mha_padding_mask
        )
        library_output, library_weights = library_mha(
            mha_input, mha_input, mha_input, attn_mask=mha_causal_mask,
            key_padding_mask=mha_padding_mask, need_weights=True, average_attn_weights=False
        )

    print(f"MHA 输出最大误差：{float((simple_output-library_output).abs().max()):.2e}")
    print(f"每头注意力权重最大误差：{float((simple_weights-library_weights).abs().max()):.2e}")


多头注意力先把 Q、K、V 投影并拆头，各头独立检索后再拼接回模型维度：

<!-- diagram:multi-head-attention -->


![架构图：多头注意力的投影、拆头、并行检索、拼接与输出投影](assets/figures/30_transformer/multi-head-attention.svg)

[TikZ 源文件](assets/figures/30_transformer/multi-head-attention.tex)


### 3.2．逐位置前馈网络（FFN）

Attention 负责位置之间的信息交换，FFN 负责各位置内部的特征变换。同一套两层 MLP 独立应用于每个位置；中间维度通常扩张到 $4D$，以增强非线性表达能力。

- **输入**：`x [N,L,E]`。
- **中间结果**：`[N,L,dim_feedforward]`。
- **输出**：`[N,L,E]`。

原论文使用 ReLU：

$$\mathrm{FFN}_{paper}(x)=W_2\,\mathrm{ReLU}(W_1x+b_1)+b_2.$$

`nn.Transformer` 的 `activation` 默认值同样是 `"relu"`，因此默认配置更接近论文。本章显式切换为 GELU，以连接后续语言模型中更平滑的激活实践：

$$\mathrm{FFN}_{chapter}(x)=W_2\,\mathrm{GELU}(W_1x+b_1)+b_2.$$

这项修改不改变 `[N,L,E] → [N,L,F] → [N,L,E]` 的形状契约，但会改变函数、参数梯度和 Checkpoint 行为。进一步的现代架构常使用带门控的 SwiGLU；它包含两条输入投影分支，不能仅靠 `nn.Transformer(activation=...)` 等价表达，需要自定义 FFN 层。本章保留两层 FFN 主线，SwiGLU 在后续现代 GPT 架构中展开。


In [ ]:
# 对每个序列位置独立应用两层前馈网络，扩展后再投影回模型维度。

class MyFeedForward(nn.Module):
    """实现逐位置两层前馈网络，保持批次和序列维度不变。"""
    def __init__(self, d_model: int, dim_feedforward: int, dropout: float = 0.0):
        """构造 E→F→E 的逐位置 MLP，并在激活后应用 dropout。"""
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, dim_feedforward),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """对 [N, S, E] 隐状态逐位置执行前馈变换并返回同形状张量。"""
        return self.net(x)

ffn = MyFeedForward(d_model=16, dim_feedforward=64)
print("FFN:", tuple(x.shape), "->", tuple(ffn(x).shape))


### 3.3．Add & Norm：残差连接、LayerNorm 与 Pre/Post-Norm

《Attention Is All You Need》架构图中的 **Add & Norm** 不是一个不可拆分的算子，而是“残差相加（Add）+ 层归一化（Layer Normalization）”的组合。论文在 Encoder 的 Self-Attention、FFN 后各使用一次，在 Decoder 的 Masked Self-Attention、Cross-Attention、FFN 后各使用一次。

#### Add：保留输入，再叠加子层学到的修正量

设输入为 $x\in\mathbb{R}^{N\times L\times E}$，Attention 或 FFN 子层为 $F$。残差连接计算 $x+F(x)$，其中 $F(x)$ 学习的是相对于原表示的增量，而不是重新构造全部表示。若当前子层没有形成有效修正，只需让 $F(x)\approx 0$，网络便接近恒等映射。其梯度包含直接通路：

$$\frac{\partial(x+F(x))}{\partial x}=I+\frac{\partial F(x)}{\partial x}.$$

Add 是逐元素相加，不是拼接或乘法，因此输入和子层输出必须同为 `[N,L,E]`。这也是多头 Attention 的输出投影以及 FFN 的第二个线性层都要回到 $E=d_{model}$ 维的结构原因。

#### Norm：对每个 Token 的特征维归一化

对任意一个 Token 向量 $z=x[n,l,:]\in\mathbb{R}^{E}$，LayerNorm 在最后一维 $E$ 上计算均值与方差：

$$\mathrm{LN}(z)=\gamma\odot\frac{z-\mu}{\sqrt{\sigma^2+\epsilon}}+\beta,$$

其中 $\mu,\sigma^2$ 是该 Token 的 $E$ 个特征的统计量，$\gamma,\beta\in\mathbb{R}^{E}$ 是可学习参数。它不跨 Batch 或序列位置统计，因此不同长度、不同 Padding 比例的样本不会像 BatchNorm 那样共享批统计量。在 PyTorch 中，这一对象对应 `nn.LayerNorm(d_model)`；输入和输出形状保持 `[N,L,E]`。

#### 论文顺序与当前实现顺序

| 结构 | 单个子层的公式 | 归一化位置 | 本章对应关系 |
|---|---|---|---|
| 原论文 Post-Norm | $y=\mathrm{LN}(x+\mathrm{Dropout}(F(x)))$ | 残差相加之后 | `norm_first=False`，也是 `nn.Transformer*` 的默认值 |
| 本章 Pre-Norm | $y=x+\mathrm{Dropout}(F(\mathrm{LN}(x)))$ | 子层计算之前 | `norm_first=True`，Stack 末尾另有一次 final LayerNorm |

原论文采用 Post-Norm，因此解读论文架构图时应按第一行理解。本章的训练闭环采用 Pre-Norm，是为了展示后续实践中常见的稳定训练顺序；研究表明归一化位置会显著影响初始化附近的梯度行为。两者包含相同组件，但运算顺序、优化性质以及 Checkpoint 参数语义不能混为一谈。参考：[原始 Transformer 论文](https://arxiv.org/abs/1706.03762)、[Pre-LN 分析论文](https://proceedings.mlr.press/v119/xiong20b.html)、[PyTorch `TransformerEncoderLayer`](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerEncoderLayer.html)。

> **现代实践边界**：部分现代 GPT 家族将 LayerNorm 换成 RMSNorm，并继续采用残差与 Pre-Norm 类顺序。PyTorch 已提供 [`nn.RMSNorm`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RMSNorm.html)，但 RMSNorm 不做均值中心化，与原论文 LayerNorm 不是同一运算。本章只建立这条演进坐标；RMSNorm 及其所在的 Decoder-only 架构在 `31_nlp_gpt.ipynb` 与 `E10_open_model.ipynb` 继续展开。


下图并列展示论文 Post-Norm 与本章 Pre-Norm。两条路径都保留 $x$ 的残差支路，区别是 LayerNorm 位于残差相加之后还是子层之前：

<!-- diagram:pre-norm-residual -->


![架构图：Post-Norm 与 Pre-Norm 的残差路径和 LayerNorm 位置对照](assets/figures/30_transformer/pre-norm-residual.svg)

[TikZ 源文件](assets/figures/30_transformer/pre-norm-residual.tex)


### 3.4．Encoder Layer 与 Encoder Stack

一个 Encoder Layer 只有两块：

1. **Self-Attention**：$Q = K = V = x$，每个源位置读取整个源序列。
2. **FFN**：对每个位置独立做非线性变换。

- **输入**：源表示 `src [N,S,E]`、`src_key_padding_mask [N,S]`。命名和形状与 `nn.Transformer` 一致。
- **输出**：编码结果 `memory [N,S,E]`。
- 堆叠 $N$ 层不会改变形状，只会逐层融合更复杂的上下文。
- MHA 已在前文完成原理实现与数值对齐，因此本节使用 `nn.MultiheadAttention`，并集中实现尚未说明的 Layer 连接方式。


In [ ]:
# 组合自注意力、前馈网络与残差结构，并按层堆叠为 Encoder。

class MyTransformerEncoderLayer(nn.Module):
    """采用 Pre-LN 结构组合自注意力、前馈网络与残差连接的编码器层。"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        """创建编码器层的自注意力、前馈、归一化和残差 dropout 模块。"""
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.ffn = MyFeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_key_padding_mask=None):
        """对源序列执行 Pre-LN 自注意力与前馈残差更新，返回 [N, S, E]。"""
        normalized = self.norm1(src)
        attn_out, _ = self.self_attn(
            normalized, normalized, normalized,
            key_padding_mask=src_key_padding_mask, need_weights=False
        )
        src = src + self.dropout(attn_out)
        src = src + self.dropout(self.ffn(self.norm2(src)))
        return src

class MyTransformerEncoder(nn.Module):
    """顺序堆叠编码器层并施加末端层归一化。"""
    def __init__(self, layer_factory, num_layers: int, d_model: int):
        """由工厂函数创建指定数量的独立编码器层及末端层归一化。"""
        super().__init__()
        self.layers = nn.ModuleList([layer_factory() for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, src_key_padding_mask=None):
        """依次传播源隐状态通过全部编码器层，并返回归一化 Memory。"""
        for layer in self.layers:
            x = layer(x, src_key_padding_mask)
        return self.final_norm(x)

encoder = MyTransformerEncoder(
    lambda: MyTransformerEncoderLayer(d_model=16, nhead=4, dim_feedforward=64, dropout=0.0),
    num_layers=2, d_model=16
)
encoder_texts = ["我爱学习。", "机器学习很有趣。"]
src_tokens_demo = my_batch_encode(encoder_texts)
encoder_embedding = nn.Embedding(VOCAB_SIZE, 16)
src_x = encoder_embedding(src_tokens_demo)
memory = encoder(src_x, my_make_key_padding_mask(src_tokens_demo))
for text in encoder_texts:
    print(f"{text} -> {my_token_labels(text)}")
print("MyTransformerEncoder:", tuple(src_x.shape), "->", tuple(memory.shape))


### 3.5．Decoder Layer 与 Cross-Attention

Decoder 每层有三块：

1. **Masked Self-Attention**：目标位置只能读取自己和过去。
2. **Cross-Attention**：`Q` 来自 Decoder，`K/V` 来自 Encoder memory；含义是“当前要生成什么”去查询“输入里有哪些相关信息”。
3. **FFN**：逐位置变换。

- **输入**：目标表示 `tgt [N,T,E]`、Encoder 记忆 `memory [N,S,E]`。
- **Mask**：`tgt_mask [T,T]`、`tgt_key_padding_mask [N,T]`、`memory_key_padding_mask [N,S]`。
- **输出**：Decoder 状态 `[N,T,E]`。
- Cross-Attention 权重形状：`[N,H,T,S]`，可以解释每个生成位置关注了哪些源位置。
- Self-Attention 不参与本节可视化，因此设置 `need_weights=False`；Cross-Attention 设置 `need_weights=True, average_attn_weights=False`。二者理论公式一致，仅诊断路径承担权重矩阵的存储成本。


In [ ]:
# 在 Decoder 中依次执行因果自注意力、Cross-Attention 和前馈网络。

class MyTransformerDecoderLayer(nn.Module):
    """组合因果自注意力、交叉注意力和前馈网络的 Pre-LN 解码器层。"""
    def __init__(self, d_model, nhead, dim_feedforward, dropout=0.1):
        """创建解码器层的自注意力、交叉注意力、前馈和三组归一化模块。"""
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.ffn = MyFeedForward(d_model, dim_feedforward, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        """依次执行因果自注意力、对 Memory 的交叉注意力及前馈更新，返回隐状态与交叉注意力权重。"""
        normalized = self.norm1(tgt)
        self_out, _ = self.self_attn(
            normalized, normalized, normalized,
            attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask, need_weights=False
        )
        tgt = tgt + self.dropout(self_out)

        query = self.norm2(tgt)
        cross_out, cross_weights = self.cross_attn(
            query, memory, memory, key_padding_mask=memory_key_padding_mask,
            need_weights=True, average_attn_weights=False
        )
        tgt = tgt + self.dropout(cross_out)
        tgt = tgt + self.dropout(self.ffn(self.norm3(tgt)))
        return tgt, cross_weights

class MyTransformerDecoder(nn.Module):
    """顺序堆叠解码器层，返回归一化隐状态与最后一层交叉注意力权重。"""
    def __init__(self, layer_factory, num_layers: int, d_model: int):
        """由工厂函数创建指定数量的独立解码器层及末端层归一化。"""
        super().__init__()
        self.layers = nn.ModuleList([layer_factory() for _ in range(num_layers)])
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x, memory, tgt_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        """传播目标隐状态通过全部解码器层，返回归一化结果和末层交叉注意力权重。"""
        cross_weights = None
        for layer in self.layers:
            x, cross_weights = layer(
                x, memory, tgt_mask, tgt_key_padding_mask, memory_key_padding_mask
            )
        return self.final_norm(x), cross_weights


一个 Encoder–Decoder 的 Decoder Layer 依次读取目标前缀和 Encoder memory：

<!-- diagram:transformer-decoder-layer -->


![架构图：Pre-Norm Transformer Decoder Layer 的三条子层与残差路径](assets/figures/30_transformer/transformer-decoder-layer.svg)

[TikZ 源文件](assets/figures/30_transformer/transformer-decoder-layer.tex)


## 4．证据验证

### 4.1．端到端翻译闭环

#### 4.1.1．Embedding + `nn.Transformer` + LM Head

Encoder Layer 与 Decoder Layer 的连接关系完成验证后，端到端训练使用 `nn.Transformer`，不再依赖原理类 `MyTransformerEncoder` / `MyTransformerDecoder`。Tokenizer、Embedding、位置编码、Mask、词表投影、损失和生成属于库主干之外的职责，仍由包装层显式实现。训练采用 **teacher forcing**：目标输出 `tgt_out` 整体右移一位，并在开头加入 `<s>` 得到 `tgt_in`。

- **Encoder 输入**：`src_tokens [N,S]`。
- **Decoder 输入**：`tgt_in [N,T]`。
- **最终输出**：`logits [N,T,V_tgt]`。
- **损失输入**：将 logits 展平为 $[N \times T, V]$，标签展平为 $[N \times T]$，忽略 `<pad>`。

工程选择：

- 使用 Pre-Norm，改善深层训练稳定性。
- Embedding 乘 $\sqrt E$，让其尺度与位置编码更匹配。
- 目标 Embedding 与输出投影共享权重，减少参数并让输入/输出语义空间一致。
- 所有 mask 在模型内统一构造，减少调用方传错形状或语义的风险。
- Encoder–Decoder 骨干直接使用 `nn.Transformer(batch_first=True, norm_first=True, activation='gelu')`。
- 包装类保留 `encode` / `decode`，便于推理时只运行一次 Encoder。
- `nn.Transformer` 不包含 Tokenizer、Embedding、位置编码、LM Head、损失或生成循环，这些仍由应用代码负责。

#### 4.1.2．显式配置参数

| 参数 | 作用与本文选择 |
|---|---|
| `padding_idx=PAD_ID` | 告诉 `nn.Embedding` 不更新 `<pad>` 这一行的梯度；Mask 仍然必须保留，两者职责不同。 |
| `batch_first=True` | 库层统一使用前文的 `[N,length,E]`，避免在模块边界频繁转置。 |
| `norm_first=True` | 对应 3.3 节的 Pre-Norm；库默认 `False` 对应原论文 Post-Norm。Pre-Norm Stack 还需要末端 LayerNorm，本包装类由 `nn.Transformer` 内部 Encoder/Decoder final norm 负责。 |
| `activation='gelu'` | 让库 FFN 使用 3.2 节说明的 GELU，而不是 `nn.Transformer` 的默认 ReLU。 |
| `bias=True` | 保留 Attention、FFN 和 LayerNorm 的可学习偏置，与前文带 $b_1,b_2$ 的 FFN 公式一致。 |
| `LM Head bias=False` | 输出层只使用与目标 Embedding 共享的权重矩阵，避免额外的词表维偏置。 |


In [ ]:
# 集中保存词表、隐藏维度、层数和特殊 token，作为模型构造契约。

# 类默认值采用 128 维、4 头、3+3 层和四倍 FFN，仅作为完整接口默认，不代表本章训练配方。
@dataclass(frozen=True)
class MyTransformerConfig:
    """集中保存最小 Encoder–Decoder Transformer 的结构与 token 配置。"""
    src_vocab_size: int
    tgt_vocab_size: int
    d_model: int = 128
    nhead: int = 4
    num_encoder_layers: int = 3
    num_decoder_layers: int = 3
    dim_feedforward: int = 512
    dropout: float = 0.1
    max_len: int = 512
    pad_id: int = PAD_ID
    bos_id: int = BOS_ID
    eos_id: int = EOS_ID
    tie_weights: bool = True


class MyTransformerForConditionalGeneration(nn.Module):
    """封装 Encoder–Decoder 主干、Embedding 和词表投影。"""
    def __init__(self, config: MyTransformerConfig):
        """依据配置构造源/目标嵌入、位置编码、编码器、解码器和词表输出层。"""
        super().__init__()
        if config.d_model % config.nhead != 0:
            raise ValueError("d_model 必须能被 nhead 整除")
        if len({config.pad_id, config.bos_id, config.eos_id}) != 3:
            raise ValueError("PAD、BOS 与 EOS 必须使用不同 ID")
        self.config = config
        self.src_embedding = nn.Embedding(config.src_vocab_size, config.d_model, padding_idx=config.pad_id)
        self.tgt_embedding = nn.Embedding(config.tgt_vocab_size, config.d_model, padding_idx=config.pad_id)
        self.position = MyPositionalEncoding(config.d_model, config.max_len, config.dropout)

        self.backbone = nn.Transformer(
            d_model=config.d_model, nhead=config.nhead,
            num_encoder_layers=config.num_encoder_layers,
            num_decoder_layers=config.num_decoder_layers,
            dim_feedforward=config.dim_feedforward, dropout=config.dropout,
            activation="gelu", batch_first=True, norm_first=True, bias=True,
        )
        self.generator = nn.Linear(config.d_model, config.tgt_vocab_size, bias=False)
        self.reset_parameters()
        if config.tie_weights:
            self.generator.weight = self.tgt_embedding.weight

    def reset_parameters(self):
        """使用 Xavier Uniform 初始化矩阵参数，并保持 PAD 向量为零。"""
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)
        with torch.no_grad():
            self.src_embedding.weight[self.config.pad_id].zero_()
            self.tgt_embedding.weight[self.config.pad_id].zero_()

    def encode(self, src_tokens):
        """把 [N, S] 源 token 编码为 [N, S, E] memory，并返回源 Padding Mask。"""
        src_key_padding_mask = my_make_key_padding_mask(src_tokens, self.config.pad_id)
        x = self.src_embedding(src_tokens) * math.sqrt(self.config.d_model)
        memory = self.backbone.encoder(
            self.position(x), src_key_padding_mask=src_key_padding_mask
        )
        return memory, src_key_padding_mask

    def decode(self, tgt_tokens, memory, memory_key_padding_mask):
        """使用因果 Mask 和 Encoder memory，返回目标位置的词表 logits。"""
        tgt_key_padding_mask = my_make_key_padding_mask(tgt_tokens, self.config.pad_id)
        tgt_mask = my_make_causal_mask(tgt_tokens.size(1), tgt_tokens.device)
        x = self.tgt_embedding(tgt_tokens) * math.sqrt(self.config.d_model)
        states = self.backbone.decoder(
            self.position(x), memory, tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
        )
        return self.generator(states)

    def forward(self, src_tokens, tgt_tokens):
        """串联 Encoder 与 Teacher Forcing Decoder，返回 `[N,T,V]` logits。"""
        memory, src_key_padding_mask = self.encode(src_tokens)
        return self.decode(tgt_tokens, memory, src_key_padding_mask)


# 训练模型采用 96 维、4 头、2+2 层和约三倍 FFN，在翻译能力与单机运行时间之间取平衡。
config = MyTransformerConfig(
    src_vocab_size=VOCAB_SIZE, tgt_vocab_size=VOCAB_SIZE, d_model=96, nhead=4,
    num_encoder_layers=2, num_decoder_layers=2, dim_feedforward=288, dropout=0.15,
    max_len=MODEL_MAX_LENGTH,
)
model = MyTransformerForConditionalGeneration(config).to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"可训练参数：{trainable:,}")

### 4.2．形状与因果性验证

库迁移后的主要风险位于张量维度、Padding Mask 与 Causal Mask。以下检查验证这些公共契约：

1. 最终输出形状为 `[N,T,V]`。
2. 在 `eval()` 下，修改未来 token 不会影响更早位置的 logits，证明 causal mask 有效。
3. `nn.Transformer` 的公共 forward 不返回 Cross-Attention 权重；正式训练和推理路径不请求权重。4.6 节仅在未编译的 `eval()` 诊断路径中临时注册观测器，并用 `finally` 确保移除；业务输出不依赖该观测器。


In [ ]:
# 只改变未来位置的输入，观察 Causal Mask 对历史位置输出的隔离作用。

model.eval()
src_test = my_batch_encode(["我爱学习。"], add_eos=True, device=DEVICE)
tgt_a = my_batch_encode(["i love learning."], add_bos=True, device=DEVICE)
tgt_b = tgt_a.clone()
tgt_b[:, -1] = my_encode("today")[-1]  # 只修改最后一个“未来”token

with torch.no_grad():
    logits_a = model(src_test, tgt_a)
    logits_b = model(src_test, tgt_b)

print("源句：我爱学习。")
print("目标：i love learning. ->", my_token_labels("i love learning."))
print("输出形状：", tuple(logits_a.shape))
print("历史位置 logits 最大差值：", float((logits_a[:, :-1] - logits_b[:, :-1]).abs().max()))

### 4.3．翻译批次与 Teacher Forcing

数据集在本节只负责提供不同长度的中文源句和英文目标句。训练时，目标序列整体右移一位：`tgt_in` 以 BOS 开始，`tgt_out` 以 EOS 结束。这样第 $t$ 个输出只根据源序列和目标前缀预测第 $t$ 个真实 token。

```text
target  = [I, love, NLP]
tgt_in  = [BOS, I, love, NLP]
tgt_out = [I, love, NLP, EOS]
```

- **批次输出**：`src [N,S]`、`tgt_in [N,T]`、`tgt_out [N,T]`。
- 不同长度序列用 PAD 对齐，损失函数忽略 PAD。
- `train` 更新参数，`validation` 选择 checkpoint，`test` 只用于最后的行为观察。

Hugging Face 的 [Translation task guide](https://huggingface.co/docs/transformers/tasks/translation) 会由数据整理器完成类似的动态 Padding。本章显式构造三个张量，是为了让目标移位和 Mask 关系保持可见。

In [ ]:
# 把变长文本编码、补齐，并显式构造 Teacher Forcing 的输入与标签。

def my_pad_token_rows(rows, pad_value, device):
    """把变长整数序列右侧补齐为 [N, Lmax] 张量。"""
    max_length = max(map(len, rows))
    result = torch.full((len(rows), max_length), pad_value, dtype=torch.long, device=device)
    for row_index, row in enumerate(rows):
        result[row_index, :len(row)] = torch.as_tensor(row, dtype=torch.long, device=device)
    return result


def my_encode_translation_pairs(selected_examples, device):
    """返回源 token、右移目标输入和目标标签三个补齐张量。"""
    src_rows, tgt_in_rows, tgt_out_rows = [], [], []
    for source_text, target_text in selected_examples:
        source_ids = my_encode(source_text, add_eos=True)
        target_ids = my_encode(target_text)
        src_rows.append(source_ids)
        tgt_in_rows.append([BOS_ID] + target_ids)
        tgt_out_rows.append(target_ids + [EOS_ID])
    return (
        my_pad_token_rows(src_rows, PAD_ID, device),
        my_pad_token_rows(tgt_in_rows, PAD_ID, device),
        my_pad_token_rows(tgt_out_rows, PAD_ID, device),
        selected_examples,
    )


def my_make_translation_batch(batch_size, examples, device):
    """有放回抽取翻译句对，并编码成训练批次。"""
    selected_examples = [random.choice(examples) for _ in range(batch_size)]
    return my_encode_translation_pairs(selected_examples, device)


src_batch, tgt_in_batch, tgt_out_batch, text_pairs = my_make_translation_batch(
    4, TRAIN_EXAMPLES, DEVICE
)
for source_text, target_text in text_pairs:
    print(f"输入：{source_text}")
    print(f"目标：{target_text} -> {my_token_labels(target_text)}")
print("src/tgt_in/tgt_out 形状：", src_batch.shape, tgt_in_batch.shape, tgt_out_batch.shape)

### 4.4．带正则化、学习率调度与早停的训练

损失仅统计非 Padding 目标位置。令 $m_{b,t}=1$ 表示位置 $(b,t)$ 的目标 Token 不是 Padding，否则为 $0$：

$$\mathcal{L}=-\frac{\sum_{b,t}m_{b,t}\log p(y_{b,t}\mid y_{b,<t},x_b)}{\sum_{b,t}m_{b,t}}$$

训练时采用 teacher forcing，因此所有目标位置可以并行计算；只有推理时才逐 token 生成。每个 epoch 无放回使用全部训练样本，并按长度分桶减少 Padding。训练预算只是上限：每个 epoch 计算一次完整 validation split 的损失，仅在验证损失获得有效改善时保存 checkpoint；连续多次没有改善便提前停止。测试集在训练和 checkpoint 选择期间完全不可见。

`AdamW` 的峰值学习率为 `1.2e-3`：首个 epoch 线性 Warmup，随后余弦衰减到峰值的 5%；更长预算不会一直以高学习率更新。`label_smoothing=0.1`、`dropout=0.15` 与 `weight_decay=0.01` 共同抑制过度自信和记忆。验证损失使用未平滑交叉熵，因而仍可解释为真实目标 Token 的负对数似然；训练损失包含 Label Smoothing，二者数值不能直接相减作为泛化间隙。另在固定训练诊断集上计算同口径未平滑损失，并与验证损失比较。

早停的 `min_delta=0.003` 表示小于该值的波动不算有效改善，`patience=3` 表示允许连续 3 次验证无改善。它们控制停止规则而不是翻译质量本身；数据、模型或评估频率改变时应重新校准。最终始终恢复验证损失最低的 checkpoint，不使用测试集选择训练步数。


In [ ]:
# 训练只读取训练集；完整验证集控制 checkpoint 与早停，测试集保持不可见。
from tqdm.auto import tqdm

def my_example_subset(examples, limit):
    """从固定划分中等距抽取确定性子集，避免诊断结果只代表文件头部。"""
    if limit <= 0:
        raise ValueError("limit 必须为正整数")
    if len(examples) <= limit:
        return list(examples)
    step = len(examples) / limit
    return [examples[int(index * step)] for index in range(limit)]


MAX_EPOCHS = 12  # 单机教学训练上限；实际轮数由完整验证集早停决定。
BATCH_SIZE = 128  # 较大 micro-batch 减少更新次数；显存不足时降为 64 并同步复核学习率。
EVAL_BATCH_SIZE = 256  # 评估不保留梯度；显存不足时可降低而不改变指标。
TRAIN_DIAGNOSTIC_SIZE = 256  # 固定训练子集仅用于观察同口径泛化间隙，不参与 checkpoint 选择。
LENGTH_BUCKET_SIZE = BATCH_SIZE * 16  # 局部长度排序范围；越大则 Padding 越少但随机混合越弱。
PEAK_LEARNING_RATE = 1.2e-3  # 96 维模型、128 样本 batch 的峰值步长；配置变化后需重调。
ADAM_BETAS = (0.9, 0.98)  # 较低 beta2 使二阶矩更快响应短程梯度变化。
ADAM_EPSILON = 1e-8  # AdamW 分母的数值稳定项。
WEIGHT_DECAY = 0.01  # 当前正则起点；应在固定数据与更新预算下比较验证损失。
LABEL_SMOOTHING = 0.1  # 降低逐字节输出过度自信；过大会损害稀有字符和精确生成。
MAX_GRAD_NORM = 1.0  # 全局梯度范数保护上限；频繁触发时应排查学习率、Batch 与异常样本。
LOSS_IGNORE_INDEX = PAD_ID  # 直接忽略目标 PAD ID；不得与通用 -100 标签哨兵混用。
EARLY_STOPPING_PATIENCE = 3  # 连续 3 个 epoch 未有效改善即停止，限制无收益训练。
VALIDATION_MIN_DELTA = 3e-3  # 小于此值视为验证噪声；改变验证规模后需重新校准。
MIN_LR_RATIO = 0.05  # 余弦末端仍保留峰值 5% 的学习率，避免训练后段完全冻结。
STEPS_PER_EPOCH = math.ceil(len(TRAIN_EXAMPLES) / BATCH_SIZE)
MAX_TRAIN_STEPS = MAX_EPOCHS * STEPS_PER_EPOCH
WARMUP_STEPS = STEPS_PER_EPOCH  # 首个 epoch 线性升温，降低随机初始化阶段的不稳定更新。
TRAIN_DIAGNOSTIC_EXAMPLES = my_example_subset(TRAIN_EXAMPLES, TRAIN_DIAGNOSTIC_SIZE)


def my_pair_length(example):
    """返回句对两侧较长的 token 数，用于减少同批 Padding。"""
    source_text, target_text = example
    return max(len(my_encode(source_text, add_eos=True)), len(my_encode(target_text)) + 1)


def my_make_length_bucketed_batches(examples):
    """先随机划分局部桶，再按长度组批并随机化批次顺序。"""
    shuffled = list(examples)
    random.shuffle(shuffled)
    batches = []
    for bucket_start in range(0, len(shuffled), LENGTH_BUCKET_SIZE):
        bucket = shuffled[bucket_start:bucket_start + LENGTH_BUCKET_SIZE]
        bucket.sort(key=my_pair_length)
        batches.extend(
            bucket[start:start + BATCH_SIZE]
            for start in range(0, len(bucket), BATCH_SIZE)
        )
    random.shuffle(batches)
    return batches


@torch.inference_mode()
def my_evaluate_translation_loss(model, examples, description):
    """在无梯度模式下计算翻译样本的 token 加权平均交叉熵损失。"""
    model.eval()
    total_loss, total_tokens = 0.0, 0
    evaluation_examples = sorted(examples, key=my_pair_length)
    batch_starts = range(0, len(evaluation_examples), EVAL_BATCH_SIZE)
    for start in tqdm(
        batch_starts, desc=description, unit="batch", leave=False, dynamic_ncols=True
    ):
        batch_examples = evaluation_examples[start:start + EVAL_BATCH_SIZE]
        src, tgt_in, tgt_out, _ = my_encode_translation_pairs(batch_examples, DEVICE)
        logits = model(src, tgt_in)
        batch_loss = F.cross_entropy(
            logits.reshape(-1, config.tgt_vocab_size),
            tgt_out.reshape(-1),
            ignore_index=LOSS_IGNORE_INDEX,
            reduction="sum",
        )
        total_loss += float(batch_loss)
        total_tokens += int(tgt_out.ne(PAD_ID).sum())
    if total_tokens == 0:
        raise RuntimeError("验证样本不包含有效目标 token")
    return total_loss / total_tokens

def my_lr_multiplier(step):
    """先线性 Warmup，再把学习率按余弦曲线衰减到指定下限。"""
    if step < WARMUP_STEPS:
        return max(step, 1) / WARMUP_STEPS
    progress = min(1.0, (step - WARMUP_STEPS) / max(1, MAX_TRAIN_STEPS - WARMUP_STEPS))
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_RATIO + (1.0 - MIN_LR_RATIO) * cosine


optimizer = torch.optim.AdamW(
    model.parameters(), lr=PEAK_LEARNING_RATE, betas=ADAM_BETAS,
    eps=ADAM_EPSILON, weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=my_lr_multiplier)
train_steps, smoothed_train_losses = [], []
evaluation_epochs, train_diagnostic_losses, validation_losses = [], [], []
best_validation_loss = float("inf")
best_train_diagnostic_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None
initial_model_state = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}

global_step = 0
training_progress = tqdm(
    total=MAX_TRAIN_STEPS, desc="训练 Encoder–Decoder", unit="step", dynamic_ncols=True
)
for epoch in range(1, MAX_EPOCHS + 1):
    epoch_batches = my_make_length_bucketed_batches(TRAIN_EXAMPLES)
    model.train()
    epoch_smoothed_loss_sum = 0.0
    epoch_token_count = 0
    for batch_examples in epoch_batches:
        src, tgt_in, tgt_out, _ = my_encode_translation_pairs(batch_examples, DEVICE)
        logits = model(src, tgt_in)
        valid_tokens = int(tgt_out.ne(PAD_ID).sum())
        loss = F.cross_entropy(
            logits.reshape(-1, config.tgt_vocab_size),
            tgt_out.reshape(-1),
            ignore_index=LOSS_IGNORE_INDEX,
            label_smoothing=LABEL_SMOOTHING,
        )
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
        optimizer.step()
        scheduler.step()
        global_step += 1
        training_progress.update(1)
        epoch_smoothed_loss_sum += loss.detach().item() * valid_tokens
        epoch_token_count += valid_tokens

    smoothed_train_loss = epoch_smoothed_loss_sum / epoch_token_count
    train_diagnostic_loss = my_evaluate_translation_loss(
        model, TRAIN_DIAGNOSTIC_EXAMPLES, f"Epoch {epoch} 训练诊断"
    )
    validation_loss = my_evaluate_translation_loss(
        model, VALIDATION_EXAMPLES, f"Epoch {epoch} 验证"
    )
    train_steps.append(global_step)
    smoothed_train_losses.append(smoothed_train_loss)
    evaluation_epochs.append(epoch)
    train_diagnostic_losses.append(train_diagnostic_loss)
    validation_losses.append(validation_loss)

    improved = validation_loss < best_validation_loss - VALIDATION_MIN_DELTA
    if improved:
        best_validation_loss = validation_loss
        best_train_diagnostic_loss = train_diagnostic_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_state = {
            name: value.detach().cpu().clone() for name, value in model.state_dict().items()
        }
    else:
        epochs_without_improvement += 1

    training_progress.set_postfix(
        train_nll=f"{train_diagnostic_loss:.4f}", validation_nll=f"{validation_loss:.4f}",
        lr=f"{scheduler.get_last_lr()[0]:.2e}", patience=f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}",
    )
    tqdm.write(
        f"epoch={epoch:2d}/{MAX_EPOCHS}  step={global_step:5d}  "
        f"train_smooth={smoothed_train_loss:.4f}  train_nll={train_diagnostic_loss:.4f}  "
        f"validation_nll={validation_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}  "
        f"patience={epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
    )
    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        tqdm.write(f"验证损失连续 {EARLY_STOPPING_PATIENCE} 个 epoch 未改善，提前停止。")
        break

training_progress.close()
if best_model_state is None:
    raise RuntimeError("训练未产生可恢复的最佳 checkpoint")
model.load_state_dict(best_model_state)
print(
    f"恢复 epoch={best_epoch} 的最佳 checkpoint："
    f"train_nll={best_train_diagnostic_loss:.4f}，validation_nll={best_validation_loss:.4f}，"
    f"gap={best_validation_loss - best_train_diagnostic_loss:+.4f}"
)

plt.figure(figsize=(8, 3.5))
plt.plot(evaluation_epochs, smoothed_train_losses, "o-", color="#315C85", label="train (smoothed CE)")
plt.plot(evaluation_epochs, train_diagnostic_losses, "o-", color="#54A24B", label="train diagnostic NLL")
plt.plot(evaluation_epochs, validation_losses, "o-", color="#E24A33", label="validation NLL")
plt.axvline(best_epoch, color="#7A5195", linestyle="--", label=f"best epoch={best_epoch}")
plt.xlabel("epoch")
plt.ylabel("交叉熵损失")
plt.title("完整验证集选择 checkpoint 并触发早停")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

### 4.5．从 `<s>` 开始自回归生成

训练能并行，推理必须自回归循环：

1. Encoder 只运行一次，得到 `memory [N,S,E]`。
2. Decoder 初始输入只有 `<s>`。
3. 取最后一个位置的 `logits [N,V]`，选出下一个 token。
4. 把 token 追加到 Decoder 输入，直到所有样本产生 `<eos>` 或达到长度上限。

本节采用贪心解码，即每步选择概率最高的 Token。生成任务可替换为 Beam Search、Top-k 或 Top-p，而无需改动 Transformer 主体。


In [ ]:
@torch.inference_mode()
def my_greedy_decode(model, src_tokens, max_new_tokens, bos_id=None, eos_id=None):
    """以自回归贪心策略从 BOS 开始生成，达到 EOS 或预算上限时停止。"""
    model.eval()
    bos_id = model.config.bos_id if bos_id is None else bos_id
    eos_id = model.config.eos_id if eos_id is None else eos_id
    memory, memory_key_padding_mask = model.encode(src_tokens)
    generated = torch.full(
        (src_tokens.size(0), 1), bos_id, dtype=torch.long, device=src_tokens.device
    )
    finished = torch.zeros(src_tokens.size(0), dtype=torch.bool, device=src_tokens.device)
    for _ in range(max_new_tokens):
        logits = model.decode(generated, memory, memory_key_padding_mask)
        next_token = logits[:, -1].argmax(dim=-1)
        next_token = torch.where(finished, torch.full_like(next_token, eos_id), next_token)
        generated = torch.cat([generated, next_token[:, None]], dim=1)
        finished |= next_token.eq(eos_id)
        if finished.all():
            break
    return generated


def my_decode_until_eos(token_ids):
    """截断 EOS 之后的 token，并解码为文本。"""
    ids = [int(token_id) for token_id in token_ids]
    if EOS_ID in ids:
        ids = ids[:ids.index(EOS_ID)]
    return my_decode(ids)


def my_tokens_through_eos(token_ids):
    """保留首个 EOS 并移除其后批量解码填入的重复 EOS。"""
    ids = [int(token_id) for token_id in token_ids]
    return ids[:ids.index(EOS_ID) + 1] if EOS_ID in ids else ids


@torch.inference_mode()
def my_evaluate_generation(model, examples):
    """批量贪心生成翻译结果，并返回逐样本 exact match 记录。"""
    source_texts = [source for source, _ in examples]
    src_tokens = my_batch_encode(source_texts, add_eos=True, device=DEVICE)
    expected = [my_encode(target) + [EOS_ID] for _, target in examples]
    generation_budget = min(MODEL_MAX_LENGTH - 1, max(map(len, expected)) + 16)
    output = my_greedy_decode(model, src_tokens, max_new_tokens=generation_budget)
    predictions, exact_matches = [], []
    for row, expected_ids in enumerate(expected):
        predicted_ids = my_tokens_through_eos(output[row, 1:].detach().cpu().tolist())
        predictions.append(my_decode_until_eos(predicted_ids))
        exact_matches.append(predicted_ids == expected_ids)
    return predictions, exact_matches, output, expected


from collections import Counter


def my_ngram_counts(text, order):
    """移除空白后，统计一条文本的字符 n-gram 多重集合。"""
    normalized = "".join(text.split())
    return Counter(
        normalized[index:index + order]
        for index in range(len(normalized) - order + 1)
    )


def my_corpus_chrf(predictions, references, max_order=6, beta=2.0):
    """按语料聚合字符 1～6 gram，计算 β=2 的 chrF 分数。"""
    if len(predictions) != len(references) or not predictions:
        raise ValueError("predictions 与 references 必须非空且长度一致")
    precisions, recalls = [], []
    for order in range(1, max_order + 1):
        matches = predicted_total = reference_total = 0
        for prediction, reference in zip(predictions, references):
            predicted_counts = my_ngram_counts(prediction, order)
            reference_counts = my_ngram_counts(reference, order)
            matches += sum((predicted_counts & reference_counts).values())
            predicted_total += sum(predicted_counts.values())
            reference_total += sum(reference_counts.values())
        if predicted_total > 0 and reference_total > 0:
            precisions.append(matches / predicted_total)
            recalls.append(matches / reference_total)
    if not precisions:
        return 0.0
    precision = sum(precisions) / len(precisions)
    recall = sum(recalls) / len(recalls)
    beta_squared = beta ** 2
    denominator = beta_squared * precision + recall
    return 0.0 if denominator == 0 else 100.0 * (1 + beta_squared) * precision * recall / denominator


def my_generation_metrics(predictions, exact_matches, examples):
    """返回教学实现的语料级 chrF 与整句 Exact Match。"""
    references = [target for _, target in examples]
    return {
        "chrf": my_corpus_chrf(predictions, references),
        "exact_match": 100.0 * sum(exact_matches) / len(exact_matches),
    }


GENERATION_DISPLAY_SIZE = 8  # 仅控制逐条打印数量。
GENERATION_METRIC_SIZE = 32  # 固定小切片用于快速趋势诊断；生产评测应覆盖完整数据与多个 seed。
TEST_GENERATION_EXAMPLES = my_example_subset(TEST_EXAMPLES, GENERATION_DISPLAY_SIZE)
TRAIN_METRIC_EXAMPLES = my_example_subset(TRAIN_EXAMPLES, GENERATION_METRIC_SIZE)
VALIDATION_METRIC_EXAMPLES = my_example_subset(VALIDATION_EXAMPLES, GENERATION_METRIC_SIZE)
TEST_METRIC_EXAMPLES = my_example_subset(TEST_EXAMPLES, GENERATION_METRIC_SIZE)

# 使用同一批测试句比较随机初始化与训练后最佳 checkpoint 的输出。
model.load_state_dict(initial_model_state)
before_predictions, _, _, _ = my_evaluate_generation(model, TEST_GENERATION_EXAMPLES)
model.load_state_dict(best_model_state)
after_predictions, display_matches, generated, expected_rows = my_evaluate_generation(
    model, TEST_GENERATION_EXAMPLES
)
train_predictions, train_matches, _, _ = my_evaluate_generation(model, TRAIN_METRIC_EXAMPLES)
validation_predictions, validation_matches, _, _ = my_evaluate_generation(
    model, VALIDATION_METRIC_EXAMPLES
)
test_predictions, test_matches, _, _ = my_evaluate_generation(model, TEST_METRIC_EXAMPLES)
split_metrics = {
    "train": my_generation_metrics(train_predictions, train_matches, TRAIN_METRIC_EXAMPLES),
    "validation": my_generation_metrics(
        validation_predictions, validation_matches, VALIDATION_METRIC_EXAMPLES
    ),
    "test": my_generation_metrics(test_predictions, test_matches, TEST_METRIC_EXAMPLES),
}
evaluation_pairs = TEST_GENERATION_EXAMPLES
src_eval = my_batch_encode(
    [source for source, _ in evaluation_pairs],
    add_eos=True,
    device=DEVICE,
)

for (source_text, target_text), before, after in zip(
    TEST_GENERATION_EXAMPLES, before_predictions, after_predictions
):
    print(f"测试输入：{source_text}")
    print(f"训练前：{before!r}")
    print(f"训练后：{after!r}")
    print(f"期望值：{target_text!r}\n")
print("固定生成切片指标（越高越好；Exact Match 仅作严格诊断）")
for split_name, metrics in split_metrics.items():
    print(
        f"{split_name:10s} chrF={metrics['chrf']:.2f}  "
        f"exact_match={metrics['exact_match']:.1f}%"
    )

#### 4.5.1．模型行为与失败案例

训练集指标只能说明优化结果，无法排除记忆。这里同时报告固定训练、验证和测试切片的 chrF 与 Exact Match：本章直接聚合字符 1～6 gram，并用 $\beta=2$ 的 F-score 强调召回率，适合观察字节级模型从乱码、英文局部结构到较完整译文的连续变化；Exact Match 要求整句完全一致，只保留为严格行为诊断，不能单独代表翻译质量。生产评测应改用 SacreBLEU 的版本化标准实现并记录签名。

- 训练与验证 chrF 同步改善，且 validation NLL 继续下降，说明增加训练仍在改善泛化。
- 训练指标继续上升而 validation NLL 不再改善或 chrF 下降，说明模型开始偏向记忆；早停应恢复更早的 checkpoint。
- 训练 Exact Match 很高而验证、测试 chrF 明显落后，是过拟合信号，不应通过继续训练追求训练集满分。
- 标点、同义表达和合法但不同的译法都会使 Exact Match 失败；因此当前指标只能作为教学实验的确定性证据，不能替代完整翻译基准、人工评审与多 Seed 置信区间。

### 4.6．Cross-Attention 对齐可视化

Cross-Attention 权重矩阵的横轴是中文源 token，纵轴是正在生成的英文目标 token。当前任务固定为 `中→英`，不添加方向前缀；模型可以学习类似 `机器学习 → machine learning` 的跨语言对齐。

- **输入**：最后一层各头权重 `[N,H,T,S]`。
- **可视化输出**：对头求平均得到 `[T,S]` 对齐图。
- 注意力图提供线索但不是严格因果解释，不能把“高权重”直接等同于“唯一原因”。

生产前向为提高性能而设置 `need_weights=False`，因此本节采用隔离的诊断路径：临时捕获最后一层 Cross-Attention 的真实 Q/K/V 与 Mask，随即移除观测器，再用同一库模块重算权重。`with_kwargs=True` 使观测器同时获得 Mask 等关键字参数；`need_weights=True` 请求返回权重；`average_attn_weights=False` 保留 $H$ 个头，不在库内提前求平均。该路径仅用于未编译的离线分析，不改变训练或在线推理。


In [ ]:
@torch.inference_mode()
def my_decode_with_last_cross_attention(model, tgt_tokens, memory, memory_key_padding_mask):
    """运行原生 decoder，并在隔离的诊断路径中重算最后一层 Cross-Attention 权重。"""
    model.eval()
    cross_attention = model.backbone.decoder.layers[-1].multihead_attn
    captured = {}

    def my_capture_inputs(module, args, kwargs):
        """前向钩子捕获 Cross-Attention 的 query、key、value 与掩码输入，不修改模块输出。"""
        if len(args) < 3:
            raise RuntimeError("Cross-Attention 观测器未收到完整的 query/key/value")
        captured.update(
            query=args[0], key=args[1], value=args[2],
            attn_mask=kwargs.get("attn_mask"),
            key_padding_mask=kwargs.get("key_padding_mask"),
            is_causal=kwargs.get("is_causal", False),
        )

    handle = cross_attention.register_forward_pre_hook(my_capture_inputs, with_kwargs=True)
    # 将可选依赖或平台能力隔离处理，不影响其余验证路径。
    try:
        logits = model.decode(tgt_tokens, memory, memory_key_padding_mask)
    finally:
        handle.remove()
    if not captured:
        raise RuntimeError("MyTransformerDecoder 前向未触发最后一层 Cross-Attention")

    _, per_head_weights = cross_attention(
        captured["query"], captured["key"], captured["value"],
        attn_mask=captured["attn_mask"],
        key_padding_mask=captured["key_padding_mask"],
        need_weights=True, average_attn_weights=False,
        is_causal=captured["is_causal"],
    )
    if per_head_weights is None:
        raise RuntimeError("Cross-Attention 权重重算失败")
    return logits, per_head_weights

memory_eval, memory_padding_eval = model.encode(src_eval)
# query 位置使用 <s> + 已生成前缀，分别预测后一个 target token。
diagnostic_tgt = generated[:, :-1]
diagnostic_logits, final_cross = my_decode_with_last_cross_attention(
    model, diagnostic_tgt, memory_eval, memory_padding_eval
)

# 选择目标序列最短的非空样本，避免 byte token 标签遮挡注意力图。
sample_index = min(range(len(evaluation_pairs)), key=lambda i: len(expected_rows[i]))
source_length = int(src_eval[sample_index].ne(PAD_ID).sum())
target_length = len(expected_rows[sample_index])
attention_map = final_cross[sample_index].mean(dim=0)[:target_length, :source_length].cpu()
source_ids = src_eval[sample_index, :source_length].cpu().tolist()
target_ids = generated[sample_index, 1:1 + target_length].cpu().tolist()
source_labels = [my_token_label(token_id) for token_id in source_ids]
target_labels = [my_token_label(token_id) for token_id in target_ids]
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(attention_map, cmap="magma", aspect="auto")
ax.set(
    xlabel="源序列位置 S（Key/Value）",
    ylabel="生成序列位置 T（Query）",
    title="最后一层 Cross-Attention：MyTransformerDecoder 如何读取 MyTransformerEncoder",
)
source_tick_step = max(1, math.ceil(len(source_labels) / 16))
target_tick_step = max(1, math.ceil(len(target_labels) / 16))
source_tick_positions = list(range(0, len(source_labels), source_tick_step))
target_tick_positions = list(range(0, len(target_labels), target_tick_step))
ax.set_xticks(source_tick_positions, labels=[source_labels[i] for i in source_tick_positions])
ax.set_yticks(target_tick_positions, labels=[target_labels[i] for i in target_tick_positions])
fig.colorbar(im, ax=ax, label="平均注意力权重")
plt.show()
print("最后一层 Cross-Attention 形状：", tuple(final_cross.shape))


## 5．迁移到生产库

### 5.1．对齐 `torch.nn.Transformer` 的职责与接口

前面的 `Transformer` 是一个可直接接收 token id 的端到端模型，内部包含 Embedding、位置编码和输出词表层；而 `torch.nn.Transformer` 本身只负责 **特征到特征** 的 Encoder–Decoder 主干，不负责 tokenizer、Embedding、位置编码、词表投影、损失或生成策略。

下图把论文架构拆成两层职责：紫色边界内是 `torch.nn.Transformer` 接管的特征主干；Embedding、位置编码、LM Head、损失、生成循环和 Mask 协议仍由端到端包装层负责。本章显式设置 `norm_first=True`、`activation='gelu'` 和 `batch_first=True`，因此不能把库默认值直接等同于论文配置。

<!-- diagram:torch-transformer-boundary -->

![架构图：端到端 Transformer 包装层与 torch.nn.Transformer 特征主干边界](assets/figures/30_transformer/torch-transformer-boundary.svg)

[TikZ 源文件](assets/figures/30_transformer/torch-transformer-boundary.tex)

为对齐 `nn.Transformer` 的职责，应将特征变换主干独立为 `MyTransformer`，而非继续扩展端到端包装类。其接口依据 [PyTorch `nn.Transformer` 官方文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.transformer.Transformer.html) 设计。

| 参数/输入 | 兼容目标 |
|---|---|
| `d_model, nhead, num_*_layers, dim_feedforward, dropout` | 与 `nn.Transformer` 同名同义 |
| `activation, layer_norm_eps, norm_first, bias` | `activation` 选 FFN 非线性；`layer_norm_eps` 防止归一化除零；`norm_first` 选 Pre/Post-Norm；`bias` 控制 Linear/LayerNorm 偏置 |
| `batch_first` | 同时支持 `[S,N,E]` 与 `[N,S,E]` |
| `src_mask` / `tgt_mask` / `memory_mask` | 布尔 mask 中 `True=禁止`；浮点 mask 直接加到注意力分数 |
| `*_key_padding_mask` | `[N,length]`，`True` 表示忽略该 key |
| `*_is_causal` | 生产库的优化提示：声明已传入的对应 mask 是 causal，它不负责创建 mask |
| 输出 | 与 `tgt` 相同布局和长度，最后一维为 `E` |

论文、原理实现与当前 PyTorch 的关系不是三份平行代码，而是一条逐级替换路径：

| 论文组件 | 本章原理对象 | `nn.Transformer` 坐标 | 当前实践中的扩展 |
|---|---|---|---|
| Token Embedding × $\sqrt{d_{model}}$ | `nn.Embedding` 包装与显式缩放 | 不属于 `nn.Transformer`，由调用方提供特征 | 继续由模型包装层负责，并与 Tokenizer、词表和权重共享策略绑定 |
| 正弦位置编码 | `MyPositionalEncoding` | `nn.Transformer` 不内置位置编码 | GPT 家族常改用 RoPE 等方案；属于 `31` / `E10` 的架构增量 |
| Scaled Dot-Product Attention | `my_scaled_dot_product_attention` | `nn.MultiheadAttention` | `F.scaled_dot_product_attention` 可按输入和后端选择融合 Kernel；自定义分数规则可进入 FlexAttention |
| Multi-Head Attention | `MyMultiHeadAttention` 与 Q/K/V 数值对齐 | Q/K/V 参数通常打包保存 | Packed projection、GQA、变长表示改变参数布局或执行效率，不能仅凭形状宣称 Checkpoint 兼容 |
| 两层 ReLU FFN | `MyFeedForward` 显式改用 GELU | `activation` 默认 `relu`，本章传入 `gelu` | SwiGLU 等门控 FFN 需要自定义层，不能只替换激活函数 |
| Add & Norm（Post-Norm） | Encoder 两条、Decoder 三条残差路径 | `norm_first=False` 对应论文；本章用 `True` 切换 Pre-Norm | RMSNorm 等会改变归一化公式与参数，需按具体模型 Config 实现 |
| Encoder–Decoder Stack | `MyTransformer` | `nn.Transformer` 是论文参考骨干 | 具体生产模型通常使用 Transformers、torchtune 等生态实现及版本化 Config |

PyTorch 官方也将 `nn.Transformer*` 定位为面向基础理解的论文参考实现，并建议现代自定义层组合 [Nested Tensor、SDPA、`torch.compile` 与 FlexAttention](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html)。这些构件主要改变张量布局、Kernel 选择和可定制性；只要公式语义未变，就应与原理实现做容差内数值或梯度对照，而不是把性能优化误写成新的 Attention 数学定义。

> 特别注意：从渐进版到本节始终采用 `nn.Transformer` 的布尔 mask 语义——**`True=屏蔽`**。它与 `F.scaled_dot_product_attention` 的布尔 mask 语义相反；官方文档也专门提示了这一点。

#### 5.1.1．原理实现与生产接口的边界

**因果性来自严格上三角 Mask**，`my_make_causal_mask(T)` 负责创建并施加该约束。生产库接口中的 `tgt_is_causal=True` 仅声明已传入的 `tgt_mask` 具有因果语义，以便 PyTorch 选择相应优化路径；该提示不负责生成 Mask。声明 `*_is_causal=True` 时若缺少对应 Mask，原理实现与当前 PyTorch 均会报错。


In [ ]:
# 统一解析激活函数参数，使原理实现与 nn.Transformer 的配置接口一致。

def my_activation_function(activation):
    """将字符串或可调用配置解析为激活函数；未知名称会触发 RuntimeError。"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if callable(activation):
        return activation
    raise ValueError("activation 只能是 'relu'、'gelu' 或可调用对象")

def my_merge_pytorch_masks(
    attn_mask, key_padding_mask, *, batch_size, num_heads, query_len, key_len,
    dtype, device, is_causal=False
):
    """把 nn.Transformer 风格 mask 合并成可加到 [N,H,L,S] 分数上的张量。"""
    if is_causal and attn_mask is None:
        raise RuntimeError(
            "is_causal=True 是对已传入 causal attn_mask 的提示，不会自动创建 mask"
        )
    merged = None

    def my_as_additive(mask):
        # 构造 mask，显式控制哪些 token 位置可以参与后续计算。
        """把布尔或浮点掩码转换为当前设备和 dtype 上的可加性掩码。"""
        mask = mask.to(device=device)
        if mask.dtype == torch.bool:
            result = torch.zeros(mask.shape, dtype=dtype, device=device)
            return result.masked_fill(mask, float("-inf"))  # True = 禁止
        if not mask.dtype.is_floating_point:
            raise TypeError("mask 必须是 bool 或浮点张量")
        return mask.to(dtype=dtype)

    if attn_mask is not None:
        if attn_mask.dim() == 2:
            if tuple(attn_mask.shape) != (query_len, key_len):
                raise ValueError(f"二维 attn_mask 应为 {(query_len, key_len)}")
            merged = my_as_additive(attn_mask)[None, None, :, :]
        elif attn_mask.dim() == 3:
            expected = (batch_size * num_heads, query_len, key_len)
            if tuple(attn_mask.shape) != expected:
                raise ValueError(f"三维 attn_mask 应为 {expected}")
            merged = my_as_additive(attn_mask).view(batch_size, num_heads, query_len, key_len)
        else:
            raise ValueError("attn_mask 只能是二维或三维")

    if key_padding_mask is not None:
        expected = (batch_size, key_len)
        if tuple(key_padding_mask.shape) != expected:
            raise ValueError(f"key_padding_mask 应为 {expected}")
        padding = my_as_additive(key_padding_mask)[:, None, None, :]
        merged = padding if merged is None else merged + padding

    return merged

class MyMultiheadAttentionReference(nn.Module):
    """数学上对齐 nn.MultiheadAttention，保留分离的 Q/K/V 投影便于阅读。"""
    def __init__(self, d_model, nhead, dropout=0.0, bias=True, device=None, dtype=None):
        """按 PyTorch 参数布局创建分离的 Q/K/V 投影和输出投影。"""
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError("d_model 必须能被 nhead 整除")
        factory_kwargs = {"device": device, "dtype": dtype}
        self.d_model, self.nhead, self.head_dim = d_model, nhead, d_model // nhead
        self.q_proj = nn.Linear(d_model, d_model, bias=bias, **factory_kwargs)
        self.k_proj = nn.Linear(d_model, d_model, bias=bias, **factory_kwargs)
        self.v_proj = nn.Linear(d_model, d_model, bias=bias, **factory_kwargs)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias, **factory_kwargs)
        self.dropout = dropout

    def my_split(self, x):
        """将批次优先隐状态重排为多头注意力表示 [N, H, L, Eh]。"""
        batch, length, _ = x.shape
        return x.view(batch, length, self.nhead, self.head_dim).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None, is_causal=False):
        """复刻 PyTorch 多头注意力的掩码合并与输出语义，返回批次优先结果。"""
        batch, query_len, _ = query.shape
        key_len = key.size(1)
        q, k, v = self.my_split(self.q_proj(query)), self.my_split(self.k_proj(key)), self.my_split(self.v_proj(value))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        additive_mask = my_merge_pytorch_masks(
            attn_mask, key_padding_mask, batch_size=batch, num_heads=self.nhead,
            query_len=query_len, key_len=key_len, dtype=scores.dtype, device=scores.device,
            is_causal=is_causal
        )
        if additive_mask is not None:
            scores = scores + additive_mask
        weights = F.softmax(scores, dim=-1)
        weights = F.dropout(weights, p=self.dropout, training=self.training)
        context = weights @ v
        context = context.transpose(1, 2).contiguous().view(batch, query_len, self.d_model)
        return self.out_proj(context)


In [ ]:
# 以下 2048、0.1 与 1e-5 等默认值对齐 PyTorch 公开接口，仅用于契约验证，不是本章训练配置。
# 实现与标准库参数布局一致的 Encoder/Decoder 参考层，便于迁移权重。

class MyTransformerEncoderLayerReference(nn.Module):
    """按 PyTorch 语义复刻编码器层，用于与官方实现逐项数值对齐。"""
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, activation=F.relu,
                 layer_norm_eps=1e-5, norm_first=False, bias=True, device=None, dtype=None):
        """创建与 torch.nn.TransformerEncoderLayer 对应的参数和归一化路径。"""
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.self_attn = MyMultiheadAttentionReference(d_model, nhead, dropout, bias, **factory_kwargs)
        self.linear1 = nn.Linear(d_model, dim_feedforward, bias=bias, **factory_kwargs)
        self.linear2 = nn.Linear(dim_feedforward, d_model, bias=bias, **factory_kwargs)
        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.dropout_ff = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = my_activation_function(activation)
        self.norm_first = norm_first

    def my_self_attention_block(self, x, mask, key_padding_mask, is_causal):
        """执行编码器自注意力并应用第一条残差支路的 dropout。"""
        return self.dropout1(self.self_attn(x, x, x, mask, key_padding_mask, is_causal))

    def my_feed_forward_block(self, x):
        """执行编码器前馈网络并应用第二条残差支路的 dropout。"""
        return self.dropout2(self.linear2(self.dropout_ff(self.activation(self.linear1(x)))))

    def forward(self, src, src_mask=None, src_key_padding_mask=None, is_causal=False):
        """依据 norm_first 配置执行编码器层，返回与输入同形状的隐状态。"""
        x = src
        if self.norm_first:
            x = x + self.my_self_attention_block(self.norm1(x), src_mask, src_key_padding_mask, is_causal)
            x = x + self.my_feed_forward_block(self.norm2(x))
        else:
            x = self.norm1(x + self.my_self_attention_block(x, src_mask, src_key_padding_mask, is_causal))
            x = self.norm2(x + self.my_feed_forward_block(x))
        return x

class MyTransformerDecoderLayerReference(nn.Module):
    """按 PyTorch 语义复刻解码器层，用于验证自注意力、交叉注意力和前馈路径。"""
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1, activation=F.relu,
                 layer_norm_eps=1e-5, norm_first=False, bias=True, device=None, dtype=None):
        """创建与 torch.nn.TransformerDecoderLayer 对应的注意力、前馈和归一化模块。"""
        super().__init__()
        factory_kwargs = {"device": device, "dtype": dtype}
        self.self_attn = MyMultiheadAttentionReference(d_model, nhead, dropout, bias, **factory_kwargs)
        self.cross_attn = MyMultiheadAttentionReference(d_model, nhead, dropout, bias, **factory_kwargs)
        self.linear1 = nn.Linear(d_model, dim_feedforward, bias=bias, **factory_kwargs)
        self.linear2 = nn.Linear(dim_feedforward, d_model, bias=bias, **factory_kwargs)
        self.norm1 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.norm2 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.norm3 = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.dropout_ff = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        self.activation = my_activation_function(activation)
        self.norm_first = norm_first

    def my_self_attention_block(self, x, mask, key_padding_mask, is_causal):
        """执行目标序列自注意力并应用第一条残差支路的 dropout。"""
        return self.dropout1(self.self_attn(x, x, x, mask, key_padding_mask, is_causal))

    def my_cross_attention_block(self, x, memory, mask, key_padding_mask, is_causal):
        """以目标隐状态查询 Encoder Memory，并应用交叉注意力支路 dropout。"""
        return self.dropout2(self.cross_attn(x, memory, memory, mask, key_padding_mask, is_causal))

    def my_feed_forward_block(self, x):
        """执行解码器前馈网络并应用第三条残差支路的 dropout。"""
        return self.dropout3(self.linear2(self.dropout_ff(self.activation(self.linear1(x)))))

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None, tgt_key_padding_mask=None,
                memory_key_padding_mask=None, tgt_is_causal=False, memory_is_causal=False):
        """依据 norm_first 配置执行解码器三条子层路径并返回更新后的目标隐状态。"""
        x = tgt
        if self.norm_first:
            x = x + self.my_self_attention_block(self.norm1(x), tgt_mask, tgt_key_padding_mask, tgt_is_causal)
            x = x + self.my_cross_attention_block(
                self.norm2(x), memory, memory_mask, memory_key_padding_mask, memory_is_causal
            )
            x = x + self.my_feed_forward_block(self.norm3(x))
        else:
            x = self.norm1(x + self.my_self_attention_block(x, tgt_mask, tgt_key_padding_mask, tgt_is_causal))
            x = self.norm2(x + self.my_cross_attention_block(
                x, memory, memory_mask, memory_key_padding_mask, memory_is_causal
            ))
            x = self.norm3(x + self.my_feed_forward_block(x))
        return x


In [ ]:
# 512 维、8 头、6+6 层、2048 维 FFN 等默认值镜像 nn.Transformer 接口；修改后不再代表默认契约。
# 组合参考层为完整 Encoder–Decoder，并集中处理 mask 与张量布局。

class MyTransformer(nn.Module):
    """接近 torch.nn.Transformer 接口的纯 Python 原理实现；内部统一采用 batch-first 计算。"""
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=2048, dropout=0.1, activation=F.relu,
                 custom_encoder=None, custom_decoder=None, layer_norm_eps=1e-5,
                 batch_first=False, norm_first=False, bias=True, device=None, dtype=None):
        """构造可替换编码器/解码器的参考 Transformer，并初始化全部权重。"""
        super().__init__()
        if custom_encoder is not None or custom_decoder is not None:
            raise NotImplementedError("原理实现不接入 custom_encoder/custom_decoder；原因见差距表")
        if d_model % nhead != 0:
            raise ValueError("d_model 必须能被 nhead 整除")
        factory_kwargs = {"device": device, "dtype": dtype}
        layer_kwargs = dict(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward, dropout=dropout,
            activation=activation, layer_norm_eps=layer_norm_eps, norm_first=norm_first,
            bias=bias, **factory_kwargs
        )
        self.encoder_layers = nn.ModuleList([
            MyTransformerEncoderLayerReference(**layer_kwargs) for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            MyTransformerDecoderLayerReference(**layer_kwargs) for _ in range(num_decoder_layers)
        ])
        self.encoder_norm = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.decoder_norm = nn.LayerNorm(d_model, eps=layer_norm_eps, bias=bias, **factory_kwargs)
        self.d_model, self.nhead, self.batch_first = d_model, nhead, batch_first
        self.reset_parameters()

    def reset_parameters(self):
        # 遍历可训练参数，统一处理梯度、更新或状态转换。
        """使用 Xavier Uniform 初始化所有维度大于一的参数。"""
        for parameter in self.parameters():
            if parameter.dim() > 1:
                nn.init.xavier_uniform_(parameter)

    @staticmethod
    def generate_square_subsequent_mask(size, device=None, dtype=None):
        """生成 PyTorch 语义的加性因果掩码：允许位置为 0，未来位置为负无穷。"""
        dtype = dtype or torch.get_default_dtype()
        return torch.triu(
            torch.full((size, size), float("-inf"), device=device, dtype=dtype), diagonal=1
        )

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None,
                src_is_causal=None, tgt_is_causal=None, memory_is_causal=False):
        """校验批次和特征维度后执行编码器与解码器，返回目标序列隐状态。"""
        if src.dim() != 3 or tgt.dim() != 3:
            raise ValueError("当前实现要求批处理的三维 src/tgt")
        if src.size(-1) != self.d_model or tgt.size(-1) != self.d_model:
            raise ValueError(f"src/tgt 最后一维必须等于 d_model={self.d_model}")
        if not self.batch_first:
            src, tgt = src.transpose(0, 1), tgt.transpose(0, 1)
        if src.size(0) != tgt.size(0):
            raise ValueError("src 与 tgt 的 batch size 必须一致")

        memory = src
        for layer in self.encoder_layers:
            memory = layer(
                memory, src_mask, src_key_padding_mask, is_causal=bool(src_is_causal)
            )
        memory = self.encoder_norm(memory)

        output = tgt
        for layer in self.decoder_layers:
            output = layer(
                output, memory, tgt_mask, memory_mask, tgt_key_padding_mask,
                memory_key_padding_mask, bool(tgt_is_causal), memory_is_causal
            )
        output = self.decoder_norm(output)
        return output if self.batch_first else output.transpose(0, 1)


### 5.2．与 `nn.Transformer` 的契约级验证

两个模型独立随机初始化时不具备数值可比性。验证过程先拆分官方模块打包保存的 Q/K/V 权重并复制到原理模块，再比较参数量、输入输出形状、容差范围内的数值一致性与因果隔离行为。这些证据说明模块组成、前向公式和 Mask 语义对齐，但不代表底层执行路径或性能相同。


In [ ]:
# 复制 nn.Transformer 权重，输出两条实现的形状、参数量和数值差异。

# 32 维、4 头与 2+2 层仅缩小数值对照成本；两条实现必须使用完全相同的配置。
core_kwargs = dict(
    d_model=32, nhead=4, num_encoder_layers=2, num_decoder_layers=2,
    dim_feedforward=96, dropout=0.0, activation="relu",
    layer_norm_eps=1e-5, batch_first=False, norm_first=False, bias=True
)
handwritten_core = MyTransformer(**core_kwargs).to(DEVICE).eval()
pytorch_core = nn.Transformer(**core_kwargs).to(DEVICE).eval()

def my_copy_mha_from_pytorch(ours, official):
    """把官方 MultiheadAttention 的合并 QKV 参数复制到分离投影实现。"""
    q_weight, k_weight, v_weight = official.in_proj_weight.chunk(3, dim=0)
    ours.q_proj.weight.copy_(q_weight)
    ours.k_proj.weight.copy_(k_weight)
    ours.v_proj.weight.copy_(v_weight)
    if official.in_proj_bias is not None:
        q_bias, k_bias, v_bias = official.in_proj_bias.chunk(3, dim=0)
        ours.q_proj.bias.copy_(q_bias)
        ours.k_proj.bias.copy_(k_bias)
        ours.v_proj.bias.copy_(v_bias)
    ours.out_proj.load_state_dict(official.out_proj.state_dict())

@torch.no_grad()
def my_copy_from_pytorch_transformer(ours, official):
    """逐层复制官方 Transformer 的注意力、前馈、归一化与末端参数。"""
    for our_layer, official_layer in zip(ours.encoder_layers, official.encoder.layers):
        my_copy_mha_from_pytorch(our_layer.self_attn, official_layer.self_attn)
        our_layer.linear1.load_state_dict(official_layer.linear1.state_dict())
        our_layer.linear2.load_state_dict(official_layer.linear2.state_dict())
        our_layer.norm1.load_state_dict(official_layer.norm1.state_dict())
        our_layer.norm2.load_state_dict(official_layer.norm2.state_dict())
    for our_layer, official_layer in zip(ours.decoder_layers, official.decoder.layers):
        my_copy_mha_from_pytorch(our_layer.self_attn, official_layer.self_attn)
        my_copy_mha_from_pytorch(our_layer.cross_attn, official_layer.multihead_attn)
        our_layer.linear1.load_state_dict(official_layer.linear1.state_dict())
        our_layer.linear2.load_state_dict(official_layer.linear2.state_dict())
        our_layer.norm1.load_state_dict(official_layer.norm1.state_dict())
        our_layer.norm2.load_state_dict(official_layer.norm2.state_dict())
        our_layer.norm3.load_state_dict(official_layer.norm3.state_dict())
    ours.encoder_norm.load_state_dict(official.encoder.norm.state_dict())
    ours.decoder_norm.load_state_dict(official.decoder.norm.state_dict())

my_copy_from_pytorch_transformer(handwritten_core, pytorch_core)

contract_source_texts = ["我爱学习。", "今天天气很好。", "注意力读取上下文。"]
contract_target_texts = ["i love learning.", "the weather is nice today.", "attention reads context."]
contract_source_ids = my_batch_encode(contract_source_texts, add_eos=True, device=DEVICE)
contract_target_ids = my_batch_encode(contract_target_texts, add_bos=True, device=DEVICE)
contract_embedding = nn.Embedding(VOCAB_SIZE, 32).to(DEVICE)
source_features = contract_embedding(contract_source_ids).transpose(0, 1)  # [S,N,E]
target_features_a = contract_embedding(contract_target_ids).transpose(0, 1)  # [T,N,E]
target_features_b = target_features_a.clone()
target_features_b[-1] = -target_features_b[-1]  # 修改最后一个未来位置
target_mask = MyTransformer.generate_square_subsequent_mask(
    target_features_a.size(0), device=DEVICE
)
source_padding = my_make_key_padding_mask(contract_source_ids)

# 关闭梯度记录，避免推理或参数更新阶段构建额外计算图。
with torch.no_grad():
    hand_a = handwritten_core(
        source_features, target_features_a, tgt_mask=target_mask,
        src_key_padding_mask=source_padding, memory_key_padding_mask=source_padding,
        tgt_is_causal=True
    )
    hand_b = handwritten_core(
        source_features, target_features_b, tgt_mask=target_mask,
        src_key_padding_mask=source_padding, memory_key_padding_mask=source_padding,
        tgt_is_causal=True
    )
    torch_output = pytorch_core(
        source_features, target_features_a, tgt_mask=target_mask,
        src_key_padding_mask=source_padding, memory_key_padding_mask=source_padding,
        tgt_is_causal=True
    )



hand_params = sum(p.numel() for p in handwritten_core.parameters())
torch_params = sum(p.numel() for p in pytorch_core.parameters())
upper_triangle = torch.ones_like(target_mask, dtype=torch.bool).triu(diagonal=1)
print("契约测试文本：", list(zip(contract_source_texts, contract_target_texts)))
print(f"原理实现输出形状：{tuple(hand_a.shape)}；PyTorch 输出形状：{tuple(torch_output.shape)}")
print(f"参数量：原理实现={hand_params:,}，nn.Transformer={torch_params:,}")
print(f"同权重输出最大误差：{float((hand_a - torch_output).abs().max()):.2e}")
print(f"未来位置改动后的历史输出最大差值：{float((hand_a[:-1] - hand_b[:-1]).abs().max()):.2e}")
print("causal mask 形状：", tuple(target_mask.shape))


### 5.3．迁移到主流 MarianMT

生产库参考采用 `Helsinki-NLP/opus-mt-zh-en`。它是中文到英文的 Marian Encoder–Decoder，包含 6 层 Encoder、6 层 Decoder、512 维隐藏状态和 8 个注意力头。这里使用 `AutoTokenizer` 与 `AutoModelForSeq2SeqLM` 展示从原理对象到 Transformers 通用推理接口的迁移。若刚升级过 Transformers 或 SentencePiece，需先重启 Notebook 内核，避免同一进程混用升级前后的自动映射模块。

Marian 使用自己的 SentencePiece Tokenizer、位置编码和特殊 token 约定，不能加载本章自建模型的 ByT5 token ID 或权重。参考模型可能在训练中接触过 OPUS/Tatoeba，因此本节只比较接口和数据流，不把单条翻译结果作为质量基准。

In [ ]:
# 使用成熟 Seq2Seq 模型展示生产库中的同一条 Encoder–Decoder 推理链。

from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList,
)

REFERENCE_MODEL_ID = "Helsinki-NLP/opus-mt-zh-en"
reference_tokenizer = AutoTokenizer.from_pretrained(
    REFERENCE_MODEL_ID,
    use_fast=False,
    trust_remote_code=False,
)
reference_model = AutoModelForSeq2SeqLM.from_pretrained(
    REFERENCE_MODEL_ID,
    use_safetensors=False,
    weights_only=True,
    trust_remote_code=False,
).to(DEVICE).eval()

reference_source_text = "注意力机制帮助模型读取上下文。"
reference_inputs = reference_tokenizer(
    [reference_source_text],
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=MODEL_MAX_LENGTH,
)
reference_inputs = {name: value.to(DEVICE) for name, value in reference_inputs.items()}


class MySeq2SeqGenerationProgress(LogitsProcessor):
    """按 Beam Search 解码步更新进度，并保持候选分数不变。"""
    def __init__(self, total):
        self.progress = tqdm(
            total=total, desc="MarianMT 翻译", unit="token-step", dynamic_ncols=True
        )

    def __call__(self, input_ids, scores):
        self.progress.update(1)
        return scores

    def close(self):
        self.progress.close()


reference_progress = MySeq2SeqGenerationProgress(64)
try:
    with torch.inference_mode():
        reference_generated = reference_model.generate(
            **reference_inputs,
            max_new_tokens=64,
            num_beams=4,
            do_sample=False,
            logits_processor=LogitsProcessorList([reference_progress]),
        )
finally:
    reference_progress.close()
reference_translation = reference_tokenizer.batch_decode(
    reference_generated, skip_special_tokens=True
)[0]

reference_config = reference_model.config
print(
    "MarianMT 结构：",
    f"encoder={reference_config.encoder_layers} 层 | ",
    f"decoder={reference_config.decoder_layers} 层 | ",
    f"d_model={reference_config.d_model} | ",
    f"heads={reference_config.encoder_attention_heads}",
)
print("输入：", reference_source_text)
print("MarianMT 输出：", reference_translation)

del reference_model, reference_inputs, reference_generated
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
elif DEVICE.type == "mps" and hasattr(torch.mps, "empty_cache"):
    torch.mps.empty_cache()

## 6．生产边界

本章的原理实现用于呈现 Transformer 数学结构，生产系统应使用 PyTorch、Transformers 或经过验证的高性能实现。自建模型与 MarianMT 的 Tokenizer、配置和权重不可互换，也不能根据本章的少量短句训练结果比较模型质量。

真实生产训练仍需要不可变数据版本、许可证与来源记录、跨语料去重、隐私和内容安全治理、独立评估集以及模型制品签名。这些属于完整生产系统的扩展边界，不构成本 notebook 的输入依赖；在正文中展开会遮蔽 Attention、Mask 与 Encoder–Decoder 主线。

### 6.1．原理实现与框架实现的差异

渐进训练主线在多头注意力完成数值对齐后即使用 `nn.MultiheadAttention`。下表比较用于建立数学与接口基线的 `MyTransformer` 与框架实现；前者不作为推荐的生产执行路径。

| 差距 | 本实现 | `nn.Transformer` | 保留差距的原因 |
|---|---|---|---|
| Q/K/V 参数布局 | 三个独立 Linear | `nn.MultiheadAttention` 通常把 Q/K/V 参数打包 | 分开更直观；参数量和数学结果等价，但 `state_dict` 键不兼容 |
| 归一化顺序与类型 | 同时实现 Post-Norm / Pre-Norm，训练闭环选择 LayerNorm + Pre-Norm | `norm_first=False/True` 均支持；参考层仍使用 LayerNorm | 现代模型可能改用 RMSNorm 或其他顺序；必须依据模型 Config，不能把 Pre-Norm、Post-Norm、RMSNorm 当作同义配置 |
| 注意力内核 | 显式 $QK^{\top} \rightarrow \operatorname{softmax} \rightarrow V$ | 可进入融合 SDPA、FlashAttention 等优化路径 | 显式代码用于理解公式；生产性能、显存占用会明显落后 |
| NestedTensor / 高 padding 优化 | 不支持 | Encoder 在满足条件时可使用 NestedTensor 快速路径 | 这属于存储布局和调度优化，不改变 Transformer 数学结构 |
| 无 Batch 输入 | 仅接收三维批处理张量 | 支持 `[S,E]` / `[T,E]` | 原理实现采用统一形状，以保持主数据流清晰 |
| causal 自动检测 | 要求显式 mask；`*_is_causal=True` 只作为已有 mask 的 hint | 可尝试从 mask 推断 hint | 本实现对齐官方的“hint 不生成 mask”语义，但不复制自动推断优化 |
| 自定义 Encoder/Decoder | 参数保留，但传入时明确报错 | 支持 `custom_encoder/custom_decoder` | 自定义模块没有统一内部布局；若要支持，需要额外定义协议和兼容测试 |
| Mask 全边界检查 | 覆盖常见二维、$N \times H$ 三维和 padding mask | 包含更多 dtype、设备、稀疏/NestedTensor 分支 | 保留关键语义和形状验证，不复制大量框架防御代码 |
| 数值逐位一致 | 不保证 | 后端、融合内核和精度策略决定结果 | 浮点运算顺序不同会产生微小误差；应验证形状、语义和容差，而非逐位相等 |
| 编译与导出 | 普通 Python 控制流 | PyTorch 内部持续适配 compile/export 和不同后端 | 框架级兼容需要长期维护，原理实现不声明未覆盖的能力 |

#### 6.1.1．对齐结论

当前 `MyTransformer` 已经在 **模块组成、参数量、主要构造参数、forward 参数、Mask 语义、Pre/Post-Norm 和输出契约** 上接近 `nn.Transformer`。主要差距集中在框架级高性能内核、特殊张量布局、自动检测和生态兼容，而不是 Transformer 数学结构。

生产系统应优先使用 PyTorch 官方模块或经过验证的高性能实现。本章的原理实现用于研究结构变化并建立等价性测试基线；Attention 公式发生变更时，应与官方模块开展梯度、数值与性能对照。

### 6.2．从论文架构到库实现的总图与边界清单

章首论文图给出 Post-LN 数学基线；下图总结本章最终采用的库实现边界。`nn.Transformer` 只替换 Encoder–Decoder 特征主干，不会自动补齐 Tokenizer、Embedding、位置编码、LM Head、损失或生成策略。

<!-- diagram:transformer-library-boundary-summary -->

![架构图：论文架构到 PyTorch 库实现的端到端责任边界](assets/figures/30_transformer/transformer-library-boundary-summary.svg)

[TikZ 源文件](assets/figures/30_transformer/transformer-library-boundary-summary.tex)

本章已经实现并连通以下组件：

- [x] Token Embedding 与位置编码
- [x] Scaled Dot-Product Attention
- [x] Padding mask 与 causal mask
- [x] Multi-Head Attention
- [x] FFN、Dropout、残差、LayerNorm
- [x] Encoder stack 与 Decoder stack
- [x] Teacher forcing 训练
- [x] 自回归贪心推理
- [x] 真实中英句对、Tokenizer 与动态 Padding
- [x] MarianMT 生产接口映射
- [x] 形状、归一化和因果性验证
- [x] `nn.Transformer` 风格的特征级核心接口
- [x] 布尔/浮点 Attention Mask 与 Key Padding Mask
- [x] 与官方模块的参数量、输出形状、数值和因果性契约对照

#### 6.2.1．大规模生产训练扩展

模型数学结构已经完整，但 Tatoeba 短句只适合机制验证，不是经过完整治理的生产训练库。大规模生产还需补充许可证与归属清单、个人信息和内容安全治理、跨语料去重、来源独立评估集、高性能 Tokenizer 与流式数据管道、混合精度、分布式训练、学习率调度、断点续训、实验追踪和自动评估。Decoder-only、KV Cache、RoPE、GQA 与现代生成接口由 GPT 和推理进阶章节展开，不纳入本章的 Encoder–Decoder 主线。